In [ ]:
# # =====================================================================
# # CELL 1 — EEG TO IMAGE: Brain Decoding via EEGFeatNet & Conditioned ControlNet
# # =====================================================================
# Complete original EEG-to-Image pipeline:
# 1. Environment & GPU Verification
# 2. Paths & Hyperparameters
# 3. EEG Data Loading & Image-Level Disjoint Splitting
# 4. Semantic Feature Decoder (EEGFeatNet) & k-NN Classification
# 5. Stable Diffusion 2.1 & Precomputed VAE Latents
# 6. EEG-Conditioned ControlNet Architecture & Subject Adapters
# 7. Training & Checkpointing Loop
# 8. Evaluation & Verification Diagnostics
# 9. Visual Reconstruction & Inference Pipeline
# Output: Reconstructed 2D Image saved to /kaggle/working/brain3d_handoff/01_eeg_decoded_image.png
# =====================================================================

import os
import shutil
import gc
import time
import json
import torch

# Unified handoff directory setup
HANDOFF_DIR = "/kaggle/working/brain3d_handoff"
os.makedirs(HANDOFF_DIR, exist_ok=True)
STAGE1_IMAGE_PATH = "/kaggle/working/brain3d_handoff/01_eeg_decoded_image.png"
LEDGER_FILE = os.path.join(HANDOFF_DIR, "execution_ledger.json")

# Establish current execution session and initialize run ledger
CURRENT_RUN_START = time.time()
initial_ledger = {
    "run_id": f"run_{int(CURRENT_RUN_START)}",
    "run_start_time": CURRENT_RUN_START,
    "stages": {}
}
with open(LEDGER_FILE, "w", encoding="utf-8") as f:
    json.dump(initial_ledger, f, indent=2)

# Purge any stale Stage 1 image artifact from previous runs
if os.path.exists(STAGE1_IMAGE_PATH):
    try:
        os.remove(STAGE1_IMAGE_PATH)
        print(f"[Cell 1 Setup] Purged stale artifact from previous run: {STAGE1_IMAGE_PATH}")
    except Exception as e:
        print(f"[Cell 1 Setup] Note removing stale artifact: {e}")

# # Graduation Project: EEG-to-Image Reconstruction
# ### Visual Stimulus Reconstruction from Brain Signals using Stable Diffusion 2.1 & ControlNet
#
# ---
#
# ### Project Overview
# The goal of this project is to decode human visual perception directly from non-invasive brain activity (EEG). When a subject views an image, 128-channel EEG signals are recorded. Our pipeline processes these brain signals and guides a latent diffusion model to reconstruct the visual image the subject was looking at.
#
# ### System Architecture
# 1. **Semantic Decoder:** A 4-layer LSTM network (`EEGFeatNet`) + k-NN classifier to predict the object category (ImageNet class) from EEG, which is then mapped into text prompt embeddings via CLIP.
# 2. **Structural ControlNet:**
#    - `SubjectLayers`: Calibrates individual skull and electrode variations across subjects into a shared space.
#    - `EEGConditionEmbedding`: 1D convolutional layers converting temporal EEG signals into 2D spatial feature maps (64x64) for ControlNet.
# 3. **Generative Backbone:** **Stable Diffusion 2.1** with frozen UNet weights to preserve generation quality and save GPU memory, training only the ControlNet branches.

# ## 1. Environment & GPU Verification
# Verify that PyTorch, Diffusers, Accelerate, and CUDA GPU acceleration are available.

# --- Stage 1 Original Cell 2 ---
import torch
import accelerate
import diffusers

from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    DDPMScheduler,
    ControlNetModel,
)

print("PyTorch Version:   ", torch.__version__)
print("Accelerate Version:", accelerate.__version__)
print("Diffusers Version: ", diffusers.__version__)
print("CUDA Available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:        ", torch.cuda.get_device_name(0))

print("Environment verified successfully.")


# ### Library Imports & Compatibility Setup
# Automatically install missing dependencies and import deep learning, image, and signal processing packages.

# --- Stage 1 Original Cell 4 ---
import os, sys, glob, time, random, pickle, urllib.request, warnings, copy
warnings.filterwarnings("ignore")

# Automatically install missing packages in Kaggle environment
required = {
    "diffusers": "diffusers",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
}
import importlib.util, subprocess
for module_name, package_name in required.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt

try:
    from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, ControlNetModel
except AttributeError:
    for module_name in [m for m in list(sys.modules) if m.startswith("diffusers")]:
        del sys.modules[module_name]
    sys.modules["torch._dynamo"] = None
    from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, ControlNetModel

from transformers import CLIPTextModel, CLIPTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. On Kaggle: Settings -> Accelerator -> GPU and rerun.")

DEVICE = torch.device("cuda")
print("Device ready:", torch.cuda.get_device_name(0))


# ## 2. Paths & Hyperparameters
# Configure dataset directories, output checkpoints, and training settings:
# - `RES = 512`: Target image resolution (512x512).
# - `BATCH_SIZE = 4` with `GRAD_ACCUM = 2`: Effective batch size of 8 while saving GPU VRAM.
# - `PROP_EMPTY = 0.5`: 50% empty-text probability to train Classifier-Free Guidance (CFG).
# - `TIME_BUDGET_H = 8.0`: Maximum training time limit to safely save checkpoints before Kaggle session timeout.

# --- Stage 1 Original Cell 6 ---
DATA_ROOT = "/kaggle/input/datasets/evolutionaw/eegcvpr40/data"
IMAGE_DIR = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train"

WORK_DIR = "/kaggle/working"
ASSET_DIR = os.path.join(WORK_DIR, "gwit_assets")
LEGACY_CHECKPOINT_PATH = os.path.join(WORK_DIR, "exact-paper-modelV00.pt")
OUT_PATH = os.path.join(WORK_DIR, "exact-paper-modelV00_6subjects.pt")
BEST_PATH = os.path.join(WORK_DIR, "best_gwit_model_6subjects.pt")

SD_IDS = [
    "stabilityai/stable-diffusion-2-1-base",
    "sd2-community/stable-diffusion-2-1-base",
]
GH = "https://raw.githubusercontent.com/LuigiSigillo/GWIT/main/src/"

# Training Hyperparameters
RES = 512
LR = 1e-5
WEIGHT_DECAY = 1e-2
MAX_GRAD_NORM = 1.0
EPOCHS = 20
PROP_EMPTY = 0.5
SEED = 42
BATCH_SIZE = 4
GRAD_ACCUM = 2
TIME_BUDGET_H = 8.0
DECODER_MIN_ACC = 0.85
DECODER_EPOCHS = 150
N_STORE_TEST = 400
SAVE_EVERY_STEPS = 1000
LATENT_BATCH_SIZE = 8
FINETUNE_DECODER_IF_NEEDED = True

os.makedirs(ASSET_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def fetch_asset(relative_path):
    """Download pretrained assets from official repo if not present locally."""
    local_path = os.path.join(ASSET_DIR, os.path.basename(relative_path))
    if not os.path.exists(local_path):
        url = GH + relative_path
        print("Downloading asset:", url)
        urllib.request.urlretrieve(url, local_path)
    return local_path

if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(f"DATA_ROOT not found: {DATA_ROOT}")
if not os.path.isdir(IMAGE_DIR):
    raise FileNotFoundError(f"IMAGE_DIR not found: {IMAGE_DIR}")

print("Paths and hyperparameters configured successfully.")


# ## 3. EEG Data Loading & Image-Level Splitting
# - **Signal Filtering:** Load 128-channel EEG recordings and extract the 440-time-step window (samples 20 to 460), corresponding to Visual Evoked Potentials (VEP).
# - **Image Linking:** Map each EEG trial to its corresponding ImageNet visual stimulus.
# - **Zero-Leakage Splitting:** Split the data (Train 70%, Val 15%, Test 15%) at the **image level** so training images never appear in validation or testing.

# --- Stage 1 Original Cell 8 ---
CKPT_FEAT = fetch_asset(
    "EEGStyleGAN-ADA/EEG2Feat/Triplet_LSTM/CVPR40/EXPERIMENT_29/bestckpt/eegfeat_all_0.9665178571428571.pth"
)
KNN_PKL = fetch_asset("gwit/dataset_EEG/knn_model.pkl")
name_map_path = fetch_asset("gwit/dataset_EEG/name_map_ID.py")

name_space = {}
with open(name_map_path, "r", encoding="utf-8") as f:
    exec(compile(f.read(), name_map_path, "exec"), name_space)

id_to_caption = name_space["id_to_caption"]
folder_label_map = name_space["folder_label_map"]
caption_to_id = {caption: int(class_id) for class_id, caption in id_to_caption.items()}
wordnet_to_id = {
    wn: caption_to_id[label]
    for wn, label in folder_label_map.items()
    if label in caption_to_id
}

# Search for data and split files
pth_files = glob.glob(os.path.join(DATA_ROOT, "**", "*.pth"), recursive=True)
split_candidates = [p for p in pth_files if "split" in os.path.basename(p).lower()]
data_candidates = [p for p in pth_files if "split" not in os.path.basename(p).lower()]
if not data_candidates:
    raise FileNotFoundError(f"No .pth files found in {DATA_ROOT}")

eeg_file = max(data_candidates, key=os.path.getsize)
split_file = None
if split_candidates:
    split_file = sorted(
        split_candidates,
        key=lambda p: ("all" not in os.path.basename(p).lower(), os.path.basename(p))
    )[0]

print("EEG File:  ", os.path.basename(eeg_file))
print("Split File:", os.path.basename(split_file) if split_file else "Dynamic Image-Disjoint Split")

# Load and filter EEG trials
data_obj = torch.load(eeg_file, map_location="cpu", weights_only=False)
image_names = data_obj["images"]
dataset_rows = data_obj["dataset"]

X_list, ORIGINAL_ROW, IMAGE_IDS, SUBJECT_IDS = [], [], [], []
for original_index, item in enumerate(tqdm(dataset_rows, desc="Reading EEG trials")):
    eeg = torch.as_tensor(item["eeg"]).float()
    if eeg.ndim != 2:
        continue
    if eeg.shape[0] != 128 and eeg.shape[1] == 128:
        eeg = eeg.t()
    if eeg.shape[0] != 128 or not (450 <= eeg.shape[1] <= 600):
        continue
    X_list.append(eeg[:, 20:460].contiguous())
    ORIGINAL_ROW.append(original_index)
    IMAGE_IDS.append(int(item["image"]))
    SUBJECT_IDS.append(int(item["subject"]))

del data_obj, dataset_rows
if not X_list:
    raise RuntimeError("No valid EEG trials loaded.")

X = torch.stack(X_list).contiguous()           # Shape: [N, 128 channels, 440 time steps]
ORIGINAL_ROW = np.asarray(ORIGINAL_ROW, dtype=np.int64)
IMAGE_IDS = np.asarray(IMAGE_IDS, dtype=np.int64)
SUBJECT_ORIGINAL = np.asarray(SUBJECT_IDS, dtype=np.int64)

# Match EEG trials to image filepaths
def find_image_path(image_name):
    folder = image_name.split("_")[0]
    candidate = os.path.join(IMAGE_DIR, folder, image_name + ".JPEG")
    if os.path.isfile(candidate):
        return candidate
    matches = glob.glob(os.path.join(IMAGE_DIR, "**", image_name + ".*"), recursive=True)
    return matches[0] if matches else None

unique_image_ids = sorted(set(IMAGE_IDS.tolist()))
IMAGE_PATHS = {image_id: find_image_path(image_names[image_id]) for image_id in unique_image_ids}
valid_mask = np.asarray([IMAGE_PATHS[image_id] is not None for image_id in IMAGE_IDS], dtype=bool)

X = X[torch.from_numpy(valid_mask)]
ORIGINAL_ROW = ORIGINAL_ROW[valid_mask]
IMAGE_IDS = IMAGE_IDS[valid_mask]
SUBJECT_ORIGINAL = SUBJECT_ORIGINAL[valid_mask]
IMAGE_WORDS = [image_names[i].split("_")[0] for i in IMAGE_IDS]
N = len(IMAGE_IDS)

# Remap subject identifiers to contiguous 0..N_SUBJECTS-1 indices
SUBJECT_VALUES = sorted(np.unique(SUBJECT_ORIGINAL).tolist())
SUBJECT_LOOKUP = {subject_id: idx for idx, subject_id in enumerate(SUBJECT_VALUES)}
SUBJECTS = np.asarray([SUBJECT_LOOKUP[s] for s in SUBJECT_ORIGINAL], dtype=np.int64)
N_SUBJECTS = len(SUBJECT_VALUES)

if N < 20:
    raise RuntimeError(f"Too few valid samples ({N}). Verify paths and data.")

rows_by_image = {}
for row_index, image_id in enumerate(IMAGE_IDS):
    rows_by_image.setdefault(int(image_id), []).append(row_index)

rng = np.random.RandomState(SEED)
if split_file is not None:
    split_obj = torch.load(split_file, map_location="cpu", weights_only=False)
    official_split = split_obj["splits"][0]
    original_to_current = {int(original_idx): row_idx for row_idx, original_idx in enumerate(ORIGINAL_ROW)}

    def map_split(key):
        return np.asarray([
            original_to_current[int(idx)]
            for idx in official_split[key]
            if int(idx) in original_to_current
        ], dtype=np.int64)

    train_rows = map_split("train")
    val_rows = map_split("val")
    test_rows = map_split("test")
else:
    all_image_ids = np.asarray(sorted(rows_by_image.keys()), dtype=np.int64)
    rng.shuffle(all_image_ids)
    n_images = len(all_image_ids)
    n_train_images = int(0.70 * n_images)
    n_val_images = int(0.15 * n_images)
    train_image_set = set(all_image_ids[:n_train_images].tolist())
    val_image_set = set(all_image_ids[n_train_images:n_train_images+n_val_images].tolist())
    test_image_set = set(all_image_ids[n_train_images+n_val_images:].tolist())

    def rows_for_image_set(image_set):
        return np.asarray([
            row for image_id in image_set for row in rows_by_image[int(image_id)]
        ], dtype=np.int64)

    train_rows = rows_for_image_set(train_image_set)
    val_rows = rows_for_image_set(val_image_set)
    test_rows = rows_for_image_set(test_image_set)

def image_overlap(rows_a, rows_b):
    return len(set(IMAGE_IDS[rows_a].tolist()) & set(IMAGE_IDS[rows_b].tolist()))

print(f"Dataset split: Total={N} | Train={len(train_rows)} | Val={len(val_rows)} | Test={len(test_rows)}")
print(f"Subjects: {N_SUBJECTS} | IDs: {SUBJECT_VALUES}")
print("Image overlap train/val (Expected 0):", image_overlap(train_rows, val_rows))
print("Image overlap train/test (Expected 0):", image_overlap(train_rows, test_rows))


# ## 4. EEG Semantic Decoder & Classification
# - `EEGFeatNet` (4-layer LSTM): Maps EEG signals into 128-dimensional semantic embeddings.
# - `k-NN Classifier`: Classifies the embedding into one of 40 visual categories.
# - **Adaptive Fine-Tuning:** If validation accuracy is under 85%, fine-tunes the LSTM on ground-truth labels to improve classification.

# --- Stage 1 Original Cell 10 ---
def normalize_eeg(batch):
    # batch: [Batch, channels=128, time=440]
    scale = batch.amax(dim=(1, 2), keepdim=True).clamp_min(1e-6) / 2.0
    return (batch - scale) / scale

class EEGFeatNet(nn.Module):
    def __init__(self, in_channels=128, n_features=128, projection_dim=128, num_layers=4):
        super().__init__()
        self.hidden_size = n_features
        self.num_layers = num_layers
        self.encoder = nn.LSTM(
            input_size=in_channels,
            hidden_size=n_features,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(n_features, projection_dim, bias=False)

    def forward(self, x):
        # x: [Batch, time=440, channels=128]
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        c0 = torch.zeros_like(h0)
        _, (hidden, _) = self.encoder(x, (h0, c0))
        return F.normalize(self.fc(hidden[-1]), dim=-1)

feature_checkpoint = torch.load(CKPT_FEAT, map_location="cpu", weights_only=False)
featnet = EEGFeatNet().to(DEVICE)
state_dict = {
    key.replace("module.", "", 1) if key.startswith("module.") else key: value
    for key, value in feature_checkpoint["model_state_dict"].items()
}
featnet.load_state_dict(state_dict)
featnet.eval()
del feature_checkpoint

with open(KNN_PKL, "rb") as f:
    knn = pickle.load(f)

@torch.no_grad()
def extract_decoder_features(rows, batch_size=128):
    feature_parts = []
    for start in range(0, len(rows), batch_size):
        row_batch = np.asarray(rows[start:start + batch_size], dtype=np.int64)
        eeg_batch = normalize_eeg(X[torch.as_tensor(row_batch)].to(DEVICE))
        features = featnet(eeg_batch.transpose(1, 2))
        feature_parts.append(features.cpu().numpy())
    if not feature_parts:
        return np.empty((0, 128), dtype=np.float32)
    return np.concatenate(feature_parts, axis=0)

FEATURES = extract_decoder_features(np.arange(N))
CAPTIONS = np.asarray(knn.predict(FEATURES), dtype=np.int64)

def decoder_accuracy(rows):
    if len(rows) == 0:
        return float("nan")
    correct, total = 0, 0
    for row in rows:
        predicted_caption = id_to_caption.get(int(CAPTIONS[row]))
        true_caption = folder_label_map.get(IMAGE_WORDS[row])
        if predicted_caption is not None and true_caption is not None:
            correct += int(predicted_caption == true_caption)
            total += 1
    return correct / total if total else 0.0

initial_decoder_acc = decoder_accuracy(val_rows)
print(f"Pretrained Decoder Validation Accuracy: {initial_decoder_acc * 100:.2f}%")

word_labels = np.asarray([wordnet_to_id.get(word, -1) for word in IMAGE_WORDS], dtype=np.int64)
valid_label_rows = np.flatnonzero((word_labels >= 0) & (word_labels < 40))

if FINETUNE_DECODER_IF_NEEDED and initial_decoder_acc < DECODER_MIN_ACC:
    valid_train_rows = np.intersect1d(train_rows, valid_label_rows, assume_unique=False)
    valid_val_rows = np.intersect1d(val_rows, valid_label_rows, assume_unique=False)
    if len(valid_train_rows) >= 128 and len(valid_val_rows) > 0:
        print("Fine-tuning EEG decoder with Cosine Softmax Loss...")
        featnet = EEGFeatNet().to(DEVICE)
        classification_head = nn.Linear(128, 40, bias=False).to(DEVICE)
        decoder_optimizer = torch.optim.AdamW(
            list(featnet.parameters()) + list(classification_head.parameters()),
            lr=1e-3,
            weight_decay=1e-2,
        )
        best_decoder_acc = -1.0
        best_decoder_state = None

        for decoder_epoch in range(DECODER_EPOCHS):
            featnet.train()
            shuffled = np.random.permutation(valid_train_rows)
            for start in range(0, len(shuffled) - 127, 128):
                batch_rows = shuffled[start:start + 128]
                eeg_batch = normalize_eeg(X[torch.as_tensor(batch_rows)].to(DEVICE))
                features = featnet(eeg_batch.transpose(1, 2))
                logits = 16.0 * F.normalize(features, dim=-1) @ F.normalize(classification_head.weight, dim=-1).T
                targets = torch.as_tensor(word_labels[batch_rows], dtype=torch.long, device=DEVICE)
                loss = F.cross_entropy(logits, targets)
                decoder_optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(featnet.parameters(), 1.0)
                decoder_optimizer.step()

            if decoder_epoch % 5 == 4 or decoder_epoch == DECODER_EPOCHS - 1:
                featnet.eval()
                current_features = extract_decoder_features(np.arange(N))
                knn_candidate = KNeighborsClassifier(n_neighbors=3)
                knn_candidate.fit(current_features[valid_train_rows], word_labels[valid_train_rows])
                val_predictions = knn_candidate.predict(current_features[valid_val_rows])
                current_acc = float(np.mean(val_predictions == word_labels[valid_val_rows]))
                if current_acc > best_decoder_acc:
                    best_decoder_acc = current_acc
                    best_decoder_state = copy.deepcopy(featnet.state_dict())

        if best_decoder_state is not None:
            featnet.load_state_dict(best_decoder_state)
            featnet.eval()
            FEATURES = extract_decoder_features(np.arange(N))
            knn = KNeighborsClassifier(n_neighbors=3)
            knn.fit(FEATURES[valid_train_rows], word_labels[valid_train_rows])
            CAPTIONS = np.asarray(knn.predict(FEATURES), dtype=np.int64)
            print(f"Best Fine-Tuned Accuracy: {best_decoder_acc * 100:.2f}%")

print(f"Final Decoder Validation Accuracy: {decoder_accuracy(val_rows) * 100:.2f}%")


# ## 5. Stable Diffusion & VAE Latent Precomputation
# - **CLIP Text Embeddings:** Compute embeddings for the 40 category prompts and empty prompt (for CFG), then remove the text encoder from GPU to free VRAM.
# - **Precomputed VAE Latents:** Encode images once into compressed latent representations (4x64x64) and store them in **CPU RAM** to avoid GPU Out-of-Memory (OOM) errors during training.

# --- Stage 1 Original Cell 12 ---
SD_ID = None
for candidate_id in SD_IDS:
    try:
        UNet2DConditionModel.load_config(candidate_id, subfolder="unet")
        SD_ID = candidate_id
        break
    except Exception as exc:
        print(f"Could not load {candidate_id}: {exc}")

if SD_ID is None:
    raise RuntimeError("Could not access Stable Diffusion 2.1 weights.")

# Precompute CLIP text embeddings and delete encoder to free VRAM
tokenizer = CLIPTokenizer.from_pretrained(SD_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(
    SD_ID, subfolder="text_encoder", torch_dtype=torch.float16
).to(DEVICE).eval()

prompts = ["image of " + id_to_caption[i] for i in range(40)] + [""]
EMPTY_PROMPT_ID = 40

with torch.no_grad():
    token_ids = tokenizer(
        prompts,
        max_length=tokenizer.model_max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).input_ids.to(DEVICE)
    TEXT_EMBEDDINGS = text_encoder(token_ids, return_dict=False)[0].half()

del text_encoder
torch.cuda.empty_cache()

# Image preprocessing and precomputed VAE latents stored on CPU RAM
image_transform = transforms.Compose([
    transforms.Resize(RES, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(RES),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

needed_image_ids = sorted(set(IMAGE_IDS[np.concatenate([train_rows, val_rows])].tolist()))
image_to_latent_row = {image_id: idx for idx, image_id in enumerate(needed_image_ids)}

vae = AutoencoderKL.from_pretrained(
    SD_ID, subfolder="vae", torch_dtype=torch.float16
).to(DEVICE).eval()
LATENT_SCALE = vae.config.scaling_factor

latent_param_chunks = []
with torch.no_grad():
    for start in tqdm(range(0, len(needed_image_ids), LATENT_BATCH_SIZE), desc="Encoding images with VAE to CPU"):
        batch_image_ids = needed_image_ids[start:start + LATENT_BATCH_SIZE]
        images = []
        for image_id in batch_image_ids:
            with Image.open(IMAGE_PATHS[int(image_id)]) as img:
                images.append(image_transform(img.convert("RGB")))
        pixel_batch = torch.stack(images).to(DEVICE, dtype=torch.float16)
        posterior_params = vae.encode(pixel_batch).latent_dist.parameters
        latent_param_chunks.append(posterior_params.cpu())

LATENT_PARAMS = torch.cat(latent_param_chunks, dim=0).contiguous()  # Stored on CPU RAM
del vae, latent_param_chunks
torch.cuda.empty_cache()

ROW_TO_LATENT = np.asarray(
    [image_to_latent_row.get(int(image_id), -1) for image_id in IMAGE_IDS],
    dtype=np.int64,
)
print("Cached VAE Latents on CPU RAM with shape:", tuple(LATENT_PARAMS.shape))
print("Stable Diffusion Backbone:", SD_ID)


# ## 6. EEG-Conditioned ControlNet Architecture
# - `SubjectLayers`: Learns a subject-specific projection matrix for each of the 6 subjects to align individual skull/electrode differences.
# - `EEGConditionEmbedding`: 1D convolutional layers transforming temporal EEG signals into 2D spatial feature maps (64x64).
# - **Frozen UNet:** Keep Stable Diffusion UNet frozen and train only the ControlNet branches.
# - `compute_loss`: Adds noise to target latents, predicts noise via ControlNet + UNet, and computes MSE loss.

# --- Stage 1 Original Cell 14 ---
class SubjectLayers(nn.Module):
    def __init__(self, channels, output_dim, n_subjects):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(n_subjects, channels, output_dim) / (channels ** 0.5))

    def forward(self, x, subject_indices):
        # x: [B, channels, time]
        _, channels, _ = x.shape
        weights = self.weights.gather(
            0,
            subject_indices.view(-1, 1, 1).expand(-1, channels, self.weights.shape[-1]),
        )
        return torch.einsum("bct,bcd->bdt", x, weights)

class EEGConditionEmbedding(nn.Module):
    def __init__(
        self,
        out_channels=320,
        channels=128,
        channel_sizes=(320, 640, 1280, 2560),
        strides=(5, 2, 2, 2),
        n_subjects=1,
    ):
        super().__init__()
        self.out_channels = out_channels
        self.subject_indices = None
        self.subj_layers = SubjectLayers(channels, channels, n_subjects)
        self.conv_in = nn.Conv1d(channels, channel_sizes[0], kernel_size=3, stride=1)
        self.blocks = nn.ModuleList()
        for i in range(len(channel_sizes) - 1):
            self.blocks.append(nn.Conv1d(channel_sizes[i], channel_sizes[i], kernel_size=3, padding=1))
            self.blocks.append(nn.Conv1d(
                channel_sizes[i],
                channel_sizes[i + 1],
                kernel_size=strides[i + 1] + 1,
                padding=1,
                stride=strides[i + 1],
            ))

    def forward(self, condition):
        x = condition.permute(0, 2, 1) if condition.shape[1] > condition.shape[2] else condition
        if self.subject_indices is None:
            raise RuntimeError("subject indices were not set before ControlNet forward.")
        if x.shape[-1] < 512:
            x = F.pad(x, (0, 512 - x.shape[-1]))
        elif x.shape[-1] > 512:
            x = x[..., :512]

        x = F.silu(self.conv_in(self.subj_layers(x, self.subject_indices)))
        for block in self.blocks:
            x = F.silu(block(x))

        if x.shape[1] % self.out_channels != 0:
            raise RuntimeError("Unexpected channel shape in EEG condition embedding.")
        x = x.reshape(
            x.shape[0],
            self.out_channels,
            x.shape[1] // self.out_channels,
            x.shape[2],
        ).permute(0, 1, 3, 2)

        x = F.pad(x, (0, max(0, 64 - x.shape[3]), 0, max(0, 64 - x.shape[2])))
        return x[:, :, :64, :64]

# Inspect existing checkpoints for smart resume / weight transfer
RESUME_CHECKPOINT = None
RESUME_SOURCE_PATH = None
SKIP_SUBJECT_WEIGHTS = False
N_SUBJECTS_MODEL = N_SUBJECTS

if os.path.exists(OUT_PATH):
    RESUME_SOURCE_PATH = OUT_PATH
elif os.path.exists(LEGACY_CHECKPOINT_PATH):
    RESUME_SOURCE_PATH = LEGACY_CHECKPOINT_PATH

if RESUME_SOURCE_PATH is not None:
    print("Found checkpoint; inspecting:", RESUME_SOURCE_PATH)
    RESUME_CHECKPOINT = torch.load(
        RESUME_SOURCE_PATH, map_location="cpu", weights_only=False
    )
    saved_control_state = RESUME_CHECKPOINT.get("controlnet", {})
    saved_subject_key = "controlnet_cond_embedding.subj_layers.weights"
    alternate_subject_key = "controlnet_cond_embedding.subject_layers.weights"
    if saved_subject_key not in saved_control_state and alternate_subject_key in saved_control_state:
        saved_subject_key = alternate_subject_key

    if saved_subject_key in saved_control_state:
        saved_n_subjects = int(saved_control_state[saved_subject_key].shape[0])
        saved_subject_values = RESUME_CHECKPOINT.get("subject_values")
        current_values = [int(v) for v in SUBJECT_VALUES]

        if saved_subject_values is not None:
            saved_subject_values = [int(v) for v in saved_subject_values]
            missing_subject_ids = sorted(set(current_values) - set(saved_subject_values))
            if missing_subject_ids:
                SKIP_SUBJECT_WEIGHTS = True
                print("Missing subject IDs in checkpoint; subject layer will be reinitialized.")
            else:
                SUBJECTS = np.asarray(
                    [saved_subject_values.index(int(v)) for v in SUBJECT_ORIGINAL],
                    dtype=np.int64,
                )
                N_SUBJECTS_MODEL = saved_n_subjects
        elif saved_n_subjects != N_SUBJECTS:
            SKIP_SUBJECT_WEIGHTS = True
            print(f"Subject count mismatch (Saved={saved_n_subjects}, Current={N_SUBJECTS}). Reinitializing subject layer.")
        else:
            N_SUBJECTS_MODEL = N_SUBJECTS

# Assemble UNet and ControlNet
unet = UNet2DConditionModel.from_pretrained(SD_ID, subfolder="unet")
controlnet = ControlNetModel.from_unet(unet)
controlnet.controlnet_cond_embedding = EEGConditionEmbedding(
    out_channels=controlnet.config.block_out_channels[0],
    n_subjects=N_SUBJECTS_MODEL,
)
unet.requires_grad_(False)
unet.to(DEVICE, dtype=torch.float16).eval()
unet.enable_gradient_checkpointing()

controlnet.to(DEVICE).train()
controlnet.enable_gradient_checkpointing()
noise_scheduler = DDPMScheduler.from_pretrained(SD_ID, subfolder="scheduler")

def compute_loss(rows, training=True):
    rows = np.asarray(rows, dtype=np.int64)
    eeg_batch = normalize_eeg(X[torch.as_tensor(rows)].to(DEVICE))
    subject_batch = torch.as_tensor(SUBJECTS[rows], dtype=torch.long, device=DEVICE)

    latent_rows = ROW_TO_LATENT[rows]
    if np.any(latent_rows < 0):
        raise RuntimeError("Missing image latent for row.")
    posterior_params = LATENT_PARAMS[torch.as_tensor(latent_rows)].to(DEVICE).float()
    mean, logvar = posterior_params.chunk(2, dim=1)
    logvar = logvar.clamp(-30.0, 20.0)
    latents = (mean + torch.exp(0.5 * logvar) * torch.randn_like(mean)) * LATENT_SCALE

    # Text dropout with PROP_EMPTY probability for Classifier-Free Guidance (CFG)
    prompt_ids = [
        EMPTY_PROMPT_ID
        if (training and random.random() < PROP_EMPTY)
        else int(CAPTIONS[row])
        for row in rows
    ]
    text_batch = TEXT_EMBEDDINGS[torch.as_tensor(prompt_ids, dtype=torch.long, device=DEVICE)]

    noise = torch.randn_like(latents)
    timesteps = torch.randint(
        0, noise_scheduler.config.num_train_timesteps,
        (len(rows),), device=DEVICE, dtype=torch.long,
    )
    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

    controlnet.controlnet_cond_embedding.subject_indices = subject_batch
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        down_residuals, mid_residual = controlnet(
            noisy_latents,
            timesteps,
            encoder_hidden_states=text_batch,
            controlnet_cond=eeg_batch,
            return_dict=False,
        )
        predicted_noise = unet(
            noisy_latents.half(),
            timesteps,
            encoder_hidden_states=text_batch,
            down_block_additional_residuals=[residual.half() for residual in down_residuals],
            mid_block_additional_residual=mid_residual.half(),
            return_dict=False,
        )[0]

    return F.mse_loss(predicted_noise.float(), noise.float())

print("ControlNet and EEG conditioning pipeline initialized.")


# ## 7. Training & Checkpointing Loop
# - Optimization via **AdamW** with learning rate `1e-5`.
# - **Mixed Precision (fp16)** via `GradScaler` with `Gradient Accumulation = 2`.
# - Validation evaluation after each epoch, saving the best model to `best_gwit_model_6subjects.pt`.

# --- Stage 1 Original Cell 16 ---
trainable_parameters = [p for p in controlnet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LR,
    betas=(0.9, 0.999),
    weight_decay=WEIGHT_DECAY,
    eps=1e-8,
)
scaler = torch.amp.GradScaler("cuda", enabled=True)

val_rng = np.random.RandomState(SEED + 1)
val_subset = val_rng.choice(val_rows, min(256, len(val_rows)), replace=False)
test_subset = np.random.RandomState(SEED + 2).choice(
    test_rows, min(N_STORE_TEST, len(test_rows)), replace=False
)

def make_checkpoint(epoch, global_step, elapsed_hours, val_loss_value=None):
    return {
        "controlnet": {k: v.detach().cpu().half() for k, v in controlnet.state_dict().items()},
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "global_step": int(global_step),
        "elapsed_hours": float(elapsed_hours),
        "val_loss": None if val_loss_value is None else float(val_loss_value),
        "n_subjects": N_SUBJECTS_MODEL,
        "subject_values": SUBJECT_VALUES,
        "sd_id": SD_ID,
        "res": RES,
        "text_emb": TEXT_EMBEDDINGS.detach().cpu(),
        "id_to_caption": id_to_caption,
        "decoder": {
            "featnet": {k: v.detach().cpu() for k, v in featnet.state_dict().items()},
            "knn_pickle": pickle.dumps(knn),
        },
        "test": {
            "eeg": X[torch.as_tensor(test_subset)].clone(),
            "subject": SUBJECTS[test_subset].copy(),
            "image_path": [IMAGE_PATHS[int(IMAGE_IDS[row])] for row in test_subset],
            "class_name": [folder_label_map.get(IMAGE_WORDS[row], IMAGE_WORDS[row]) for row in test_subset],
        },
        "hparams": {
            "LR": LR, "EPOCHS": EPOCHS, "PROP_EMPTY": PROP_EMPTY,
            "BATCH_SIZE": BATCH_SIZE, "GRAD_ACCUM": GRAD_ACCUM,
            "split_sizes": (len(train_rows), len(val_rows), len(test_rows)),
        },
    }

@torch.no_grad()
def evaluate_validation_loss():
    controlnet.eval()
    losses = []
    cpu_rng_state = torch.random.get_rng_state()
    cuda_rng_state = torch.cuda.get_rng_state_all()
    random_state = random.getstate()
    np_state = np.random.get_state()
    torch.manual_seed(SEED + 123)
    torch.cuda.manual_seed_all(SEED + 123)
    random.seed(SEED + 123)
    np.random.seed(SEED + 123)
    for start in range(0, len(val_subset), BATCH_SIZE):
        rows = val_subset[start:start + BATCH_SIZE]
        if len(rows):
            losses.append(compute_loss(rows, training=False).item())
    torch.random.set_rng_state(cpu_rng_state)
    torch.cuda.set_rng_state_all(cuda_rng_state)
    random.setstate(random_state)
    np.random.set_state(np_state)
    controlnet.train()
    return float(np.mean(losses)) if losses else float("inf")

start_epoch = 1
global_step = 0
if RESUME_SOURCE_PATH is not None:
    print("Loading weights from:", RESUME_SOURCE_PATH)
    checkpoint = RESUME_CHECKPOINT or torch.load(RESUME_SOURCE_PATH, map_location="cpu", weights_only=False)
    saved_state = {}
    for key, value in checkpoint["controlnet"].items():
        normalized_key = key.replace(
            "controlnet_cond_embedding.subject_layers.",
            "controlnet_cond_embedding.subj_layers."
        )
        saved_state[normalized_key] = value

    current_state = controlnet.state_dict()
    subject_key = "controlnet_cond_embedding.subj_layers.weights"
    compatible_state = {}
    for key, saved_tensor in saved_state.items():
        if key in current_state and tuple(saved_tensor.shape) == tuple(current_state[key].shape):
            if key == subject_key and SKIP_SUBJECT_WEIGHTS:
                continue
            compatible_state[key] = saved_tensor

    controlnet.load_state_dict(compatible_state, strict=False)
    if not SKIP_SUBJECT_WEIGHTS and "optimizer" in checkpoint:
        try:
            optimizer.load_state_dict(checkpoint["optimizer"])
            start_epoch = int(checkpoint.get("epoch", 0)) + 1
            global_step = int(checkpoint.get("global_step", 0))
            print("Restored optimizer state. Resuming from epoch:", start_epoch)
        except Exception:
            print("Using fresh optimizer state.")
    else:
        print("Compatible non-subject weights loaded successfully.")
    del checkpoint

best_val_loss = float("inf")
if os.path.exists(BEST_PATH):
    best_cp = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
    best_val_loss = float(best_cp.get("val_loss", float("inf")))
    del best_cp

run_start_time = time.time()
stop_training = False

for epoch in range(start_epoch, EPOCHS + 1):
    controlnet.train()
    shuffled_rows = np.random.permutation(train_rows)
    usable_count = (len(shuffled_rows) // BATCH_SIZE) * BATCH_SIZE
    shuffled_rows = shuffled_rows[:usable_count]
    epoch_losses = []
    optimizer.zero_grad(set_to_none=True)
    pending_micro_batches = 0

    progress = tqdm(
        range(0, len(shuffled_rows), BATCH_SIZE),
        desc=f"Epoch {epoch:02d}/{EPOCHS:02d}",
    )

    for batch_start in progress:
        rows = shuffled_rows[batch_start:batch_start + BATCH_SIZE]
        loss = compute_loss(rows, training=True)
        scaler.scale(loss / GRAD_ACCUM).backward()
        epoch_losses.append(float(loss.item()))
        pending_micro_batches += 1

        is_accumulation_boundary = (pending_micro_batches == GRAD_ACCUM)
        is_last_batch = (batch_start + BATCH_SIZE) >= len(shuffled_rows)

        if is_accumulation_boundary or is_last_batch:
            if pending_micro_batches < GRAD_ACCUM:
                scaler.unscale_(optimizer)
                correction = GRAD_ACCUM / pending_micro_batches
                for parameter in trainable_parameters:
                    if parameter.grad is not None:
                        parameter.grad.mul_(correction)
            else:
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            pending_micro_batches = 0
            global_step += 1

            if global_step % SAVE_EVERY_STEPS == 0:
                cp = make_checkpoint(epoch, global_step, (time.time() - run_start_time) / 3600)
                torch.save(cp, OUT_PATH)
                del cp

        progress.set_postfix(train_loss=f"{np.mean(epoch_losses[-20:]):.4f}", step=global_step)

        if (time.time() - run_start_time) >= TIME_BUDGET_H * 3600:
            stop_training = True
            break

    current_val_loss = evaluate_validation_loss()
    mean_train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    elapsed_hours = (time.time() - run_start_time) / 3600
    print(
        f"Epoch {epoch:02d} | train loss={mean_train_loss:.5f} | "
        f"val loss={current_val_loss:.5f} | elapsed={elapsed_hours:.2f} h"
    )

    cp = make_checkpoint(epoch, global_step, elapsed_hours, current_val_loss)
    torch.save(cp, OUT_PATH)

    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        torch.save(cp, BEST_PATH)
        print(f"Saved BEST model checkpoint: val_loss={best_val_loss:.5f}")

    del cp
    torch.cuda.empty_cache()

    if stop_training:
        print("Time budget reached. Checkpoint saved safely.")
        break

print("Training finished.")
print("Latest checkpoint:", OUT_PATH)
print("Best checkpoint:", BEST_PATH)


# ## 8. Evaluation & Verification Diagnostics
# Print quantitative metrics (decoder accuracy, dataset split sizes, image overlap verification) and load the best saved checkpoint.

# --- Stage 1 Original Cell 18 ---
print("Decoder validation class accuracy:", f"{decoder_accuracy(val_rows):.4f}")
print("Split sizes:", {
    "train": len(train_rows),
    "validation": len(val_rows),
    "test": len(test_rows),
})
print("Unique image overlap (Zero expected):", {
    "train-validation": image_overlap(train_rows, val_rows),
    "train-test": image_overlap(train_rows, test_rows),
    "validation-test": image_overlap(val_rows, test_rows),
})
print("Latest checkpoint exists:", os.path.isfile(OUT_PATH))
print("Best checkpoint exists:", os.path.isfile(BEST_PATH))

# Load best checkpoint weights
checkpoint_to_load = BEST_PATH if os.path.exists(BEST_PATH) else (OUT_PATH if os.path.exists(OUT_PATH) else None)
if checkpoint_to_load is not None:
    checkpoint = torch.load(checkpoint_to_load, map_location="cpu", weights_only=False)
    controlnet.load_state_dict(checkpoint["controlnet"], strict=True)
    controlnet.eval()
    print("Loaded ControlNet checkpoint successfully from:", os.path.basename(checkpoint_to_load))
    print("  - Best Epoch:", checkpoint.get("epoch"))
    print("  - Best Validation Loss:", checkpoint.get("val_loss"))


# ## 9. Visual Reconstruction & Inference Pipeline
# **End-to-End Demonstration:**
# 1. Select an unseen EEG sample from the test set.
# 2. Predict the semantic prompt via k-NN and condition ControlNet with the EEG signal.
# 3. Run reverse diffusion sampling over 30 steps with Classifier-Free Guidance (CFG).
# 4. Decode latents with VAE and display a 3-panel comparison:
#    **(Raw EEG Waveform + Ground Truth Image + AI Reconstructed Image from Brain Signals)**.

# --- Stage 1 Original Cell 20 (Visual Reconstruction Function) ---
@torch.no_grad()
def reconstruct_eeg_to_image(
    test_idx,
    num_inference_steps=30,
    guidance_scale=7.5,
    seed=42,
    save_fig_path=None
):
    """Reconstruct image from EEG trial and plot side-by-side comparison."""
    if seed is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    row = int(test_idx)
    raw_eeg = X[row].clone()
    subject_id = SUBJECTS[row]
    gt_image_path = IMAGE_PATHS[int(IMAGE_IDS[row])]
    true_label = folder_label_map.get(IMAGE_WORDS[row], "Unknown")
    pred_label_id = int(CAPTIONS[row])
    pred_label = id_to_caption.get(pred_label_id, "Unknown")

    # 1. Prepare conditioning
    eeg_batch = normalize_eeg(raw_eeg.unsqueeze(0)).to(DEVICE)
    subject_batch = torch.as_tensor([subject_id], dtype=torch.long, device=DEVICE)
    controlnet.controlnet_cond_embedding.subject_indices = subject_batch

    cond_text = TEXT_EMBEDDINGS[pred_label_id].unsqueeze(0).to(DEVICE)
    uncond_text = TEXT_EMBEDDINGS[EMPTY_PROMPT_ID].unsqueeze(0).to(DEVICE)

    # 2. Setup noise scheduler
    scheduler = copy.deepcopy(noise_scheduler)
    scheduler.set_timesteps(num_inference_steps, device=DEVICE)

    # 3. Initial noise latents
    latents = torch.randn((1, 4, RES // 8, RES // 8), device=DEVICE, dtype=torch.float32)
    latents = latents * scheduler.init_noise_sigma

    controlnet.eval()
    unet.eval()

    # 4. Reverse Diffusion Sampling Loop
    for t in tqdm(scheduler.timesteps, desc=f"Reconstructing #{row}", leave=False):
        latent_model_input = scheduler.scale_model_input(latents, t)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            down_residuals, mid_residual = controlnet(
                latent_model_input,
                t,
                encoder_hidden_states=cond_text,
                controlnet_cond=eeg_batch,
                return_dict=False
            )

            noise_pred_cond = unet(
                latent_model_input.half(),
                t,
                encoder_hidden_states=cond_text,
                down_block_additional_residuals=[r.half() for r in down_residuals],
                mid_block_additional_residual=mid_residual.half(),
                return_dict=False
            )[0]

            noise_pred_uncond = unet(
                latent_model_input.half(),
                t,
                encoder_hidden_states=uncond_text,
                down_block_additional_residuals=[r.half() for r in down_residuals],
                mid_block_additional_residual=mid_residual.half(),
                return_dict=False
            )[0]

            # Apply Classifier-Free Guidance (CFG)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)

        latents = scheduler.step(noise_pred.float(), t, latents).prev_sample

    # 5. Decode latents to RGB image via VAE
    vae = AutoencoderKL.from_pretrained(SD_ID, subfolder="vae", torch_dtype=torch.float16).to(DEVICE).eval()
    with torch.no_grad():
        decoded_image = vae.decode(latents.half() / LATENT_SCALE).sample
        decoded_image = (decoded_image / 2.0 + 0.5).clamp(0.0, 1.0)
        decoded_image = decoded_image.cpu().permute(0, 2, 3, 1).float().numpy()[0]
        reconstructed_pil = Image.fromarray((decoded_image * 255).astype(np.uint8))

    del vae
    torch.cuda.empty_cache()

    # 6. Plot 3-panel comparison figure
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=130)

    # (a) Raw EEG Waveform
    time_axis = np.arange(raw_eeg.shape[1])
    for ch in [0, 16, 32, 64, 96, 127]:
        axes[0].plot(time_axis, raw_eeg[ch].cpu().numpy(), alpha=0.7, label=f"Ch {ch+1}")
    axes[0].set_title(f"Visual Evoked EEG\n(Subject: {SUBJECT_ORIGINAL[row]})", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Time Steps (440)", fontsize=9)
    axes[0].set_ylabel("Amplitude", fontsize=9)
    axes[0].grid(True, linestyle="--", alpha=0.4)
    axes[0].legend(loc="upper right", fontsize=7, ncol=2)

    # (b) Ground Truth Image
    if gt_image_path and os.path.isfile(gt_image_path):
        axes[1].imshow(Image.open(gt_image_path).convert("RGB"))
    else:
        axes[1].text(0.5, 0.5, "Image not found", ha="center")
    axes[1].set_title(f"Ground Truth Image\nClass: {true_label}", fontsize=11, fontweight="bold")
    axes[1].axis("off")

    # (c) Reconstructed Image from Brain
    axes[2].imshow(reconstructed_pil)
    match_tag = "✓ Match" if true_label == pred_label else "≈ Semantic"
    axes[2].set_title(f"Reconstructed from Brain\nDecoded: {pred_label} ({match_tag})", fontsize=11, fontweight="bold")
    axes[2].axis("off")

    plt.tight_layout()
    if save_fig_path:
        plt.savefig(save_fig_path, bbox_inches="tight")
    plt.show()

    return reconstructed_pil



# =====================================================================
# Strict Visual Reconstruction & Stage 1 Artifact Export
# =====================================================================
# Run reconstruction on the first test sample
if 'test_subset' not in locals() or len(test_subset) == 0:
    raise RuntimeError(
        "[Cell 1 Error] Test subset is empty or undefined. No EEG samples available for visual reconstruction. "
        "Verify dataset loading and image-level disjoint splitting."
    )

print(f"Generating visual reconstruction for sample index {test_subset[0]}...")
try:
    sample_img = reconstruct_eeg_to_image(
        test_idx=test_subset[0],
        num_inference_steps=30,
        guidance_scale=7.5,
        seed=42
    )
except Exception as exc:
    raise RuntimeError(
        f"[Cell 1 Error] EEG-to-image reconstruction inference failed during current execution: {exc}"
    ) from exc

if sample_img is None:
    raise RuntimeError(
        "[Cell 1 Error] EEG-to-image reconstruction produced no image (sample_img is None). "
        "Inference failed during current execution."
    )

# Save actual reconstructed image
sample_img.save(STAGE1_IMAGE_PATH)
if not os.path.exists(STAGE1_IMAGE_PATH) or os.path.getsize(STAGE1_IMAGE_PATH) == 0:
    raise RuntimeError(
        f"[Cell 1 Error] Failed to verify newly saved image at {STAGE1_IMAGE_PATH}."
    )

file_size = os.path.getsize(STAGE1_IMAGE_PATH)
dims = list(sample_img.size)
print(f"\n✓ [Cell 1 Output] Successfully saved reconstructed EEG image to: {STAGE1_IMAGE_PATH} ({file_size/1024:.1f} KB, dimensions: {dims})")

# Record Cell 1 completion in ledger
try:
    with open(LEDGER_FILE, "r", encoding="utf-8") as f:
        ledger = json.load(f)
    ledger["stages"]["cell_1"] = {
        "status": "COMPLETED",
        "source_var": "sample_img",
        "dest_var": "image_url (Cell 2)",
        "output_path": STAGE1_IMAGE_PATH,
        "dimensions": dims,
        "file_size": file_size,
        "timestamp": os.path.getmtime(STAGE1_IMAGE_PATH)
    }
    with open(LEDGER_FILE, "w", encoding="utf-8") as f:
        json.dump(ledger, f, indent=2)
except Exception as e:
    print(f"[Cell 1 Warning] Could not update ledger: {e}")

# Clean up GPU VRAM for Stage 2
for v in ['controlnet', 'unet', 'vae', 'noise_scheduler', 'scheduler']:
    if v in locals():
        try: del locals()[v]
        except Exception: pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"[Cell 1 VRAM] GPU memory freed. Allocated: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")

In [ ]:
# # =====================================================================
# # CELL 2 — IMAGE TO LLM PROMPT: 3D-Aware Semantic Reasoning via LLaVA-1.5-7B
# # =====================================================================
# Receives the actual image from Cell 1: /kaggle/working/brain3d_handoff/01_eeg_decoded_image.png
# Analyzes category, geometry, materials, and parts using multi-turn visual questions.
# Applies anti-hallucination verification rules and 3D studio background constraints.
# Saves actual generated prompt to: /kaggle/working/brain3d_handoff/02_mllm_3d_prompt.txt
# =====================================================================

import subprocess
import sys
import os
import gc
import json
import torch

# Install required quantization and model libraries
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes", "accelerate"])

STAGE1_IMAGE_PATH = "/kaggle/working/brain3d_handoff/01_eeg_decoded_image.png"
STAGE2_PROMPT_PATH = "/kaggle/working/brain3d_handoff/02_mllm_3d_prompt.txt"
LEDGER_FILE = "/kaggle/working/brain3d_handoff/execution_ledger.json"

# Purge any stale prompt artifact from previous runs
if os.path.exists(STAGE2_PROMPT_PATH):
    try: os.remove(STAGE2_PROMPT_PATH)
    except Exception: pass

# STRICT RULE: Must receive the actual image produced by Cell 1 without fake fallbacks
if not os.path.exists(STAGE1_IMAGE_PATH) or os.path.getsize(STAGE1_IMAGE_PATH) == 0:
    raise FileNotFoundError(
        f"[Cell 2 Error] Input image from Cell 1 not found or empty at: {STAGE1_IMAGE_PATH}.\n"
        "Cell 1 must complete successfully during current execution before running Cell 2."
    )

print(f"✓ [Cell 2 Input] Verified actual input image from Cell 1: {STAGE1_IMAGE_PATH} ({os.path.getsize(STAGE1_IMAGE_PATH)/1024:.1f} KB)")

import gc
import os
import re
import torch
import requests
from io import BytesIO
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

# ==========================================
# 1. تحميل النموذج مع الضغط لتوفير الذاكرة
# ==========================================
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_id = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    low_cpu_mem_usage=True,
    device_map="auto"
)

# ==========================================
# 2. إعدادات ثابتة
# ==========================================
WHITE_BG_SUFFIX = (
    "The subject is fully visible, centered and completely isolated on a solid plain "
    "pure white background, lit by soft neutral studio lighting with even illumination, "
    "no cast shadows, no reflections of the environment, sharp focus, ultra high resolution, "
    "clean 3D reconstruction ready render, three-quarter front view."
)

# أسئلة مخصصة لكل نوع: (السؤال، أقصى عدد tokens)
QUESTION_SETS = {
    "person": [
        ("Describe the body build and proportions of this {s} in one short sentence.", 50),
        ("Describe the head of this {s}: hair color, hair length, hairstyle, facial hair, eye color, skin tone. "
         "One short sentence.", 60),
        ("Describe every clothing item of this {s} with its exact color, including top, bottom and shoes if visible. "
         "One or two short sentences. Do not add anything you cannot clearly see.", 80),
        ("List any accessories clearly visible on this {s} (glasses, watch, jewelry, hat, bag, belt). "
         "If there are none, answer: none.", 40),
        ("What fabric or material textures are visible on the clothing of this {s}? "
         "Use words like matte, glossy, cotton, wool, leather, denim, knit. One short sentence.", 50),
        ("Describe the body pose of this {s}: standing, sitting, leaning or other, and where the arms and legs are. "
         "One short sentence.", 70),
    ],
    "animal": [
        ("What is the exact species and breed of this {s}? One short sentence.", 30),
        ("Describe the body shape, size and proportions of this {s} in one short sentence.", 50),
        ("Describe the fur, skin, feathers or scales of this {s}: exact colors, patterns and markings. "
         "One or two short sentences.", 80),
        ("Describe the head of this {s}: eyes, ears, nose, snout or beak, mouth. One short sentence.", 60),
        ("Describe the legs, paws, wings and tail of this {s}. One short sentence.", 50),
        ("Describe the pose of this {s}: standing, sitting, lying, running, and where the head is facing. "
         "One short sentence.", 60),
    ],
    "object": [
        ("Describe the overall shape, geometry and proportions of this {s} in one or two short sentences.", 70),
        ("List the main visible parts and components of this {s} and how they connect. "
         "One or two short sentences. Only mention parts you can clearly see.", 80),
        ("What is the exact color of each part of this {s}? One or two short sentences.", 70),
        ("What materials and surface finishes does this {s} have? "
         "Use words like glossy, matte, metal, plastic, wood, glass, fabric, leather, ceramic. "
         "One short sentence.", 50),
        ("Describe the fine details of this {s}: patterns, stitching, buttons, engravings, edges, wear. "
         "Only what you can clearly see. One short sentence.", 70),
        ("Describe the orientation of this {s}: which side faces the camera and how it is positioned. "
         "One short sentence.", 50),
    ],
}

# كلمات بيهلوس فيها النموذج كتير -> نتأكد منها بسؤال yes/no
VERIFY_ITEMS = {
    r"\b(tie|necktie|bow tie)\b": "a necktie or bow tie",
    r"\b(glasses|sunglasses|eyeglasses)\b": "glasses",
    r"\b(hat|cap|beanie)\b": "a hat or cap",
    r"\b(watch|wristwatch)\b": "a wristwatch",
    r"\b(necklace|chain|pendant)\b": "a necklace",
    r"\b(earrings?)\b": "earrings",
    r"\b(bag|backpack|purse)\b": "a bag",
    r"\b(belt)\b": "a belt",
    r"\b(logo|text|lettering|writing|brand)\b": "a logo or text",
    r"\b(scarf)\b": "a scarf",
    r"\b(gloves?)\b": "gloves",
}

# ==========================================
# 3. دوال مساعدة
# ==========================================
def load_image(source):
    """يقبل رابط (http/https) أو مسار ملف على Kaggle"""
    if source.startswith("http"):
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(source, headers=headers, timeout=15)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert("RGB")
    if os.path.exists(source):
        return Image.open(source).convert("RGB")
    raise ValueError("المصدر مش رابط صحيح ولا مسار ملف موجود")


def ask_llava(image, question, max_new_tokens=200):
    full_prompt = f"USER: <image>\n{question}\nASSISTANT:"
    inputs = processor(text=full_prompt, images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        generate_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1
        )
    text = processor.batch_decode(generate_ids, skip_special_tokens=True)[0]
    del inputs, generate_ids
    torch.cuda.empty_cache()
    return text.split("ASSISTANT:")[-1].strip()


def yes_no(image, question):
    ans = ask_llava(image, question + " Answer only yes or no.", max_new_tokens=3)
    return ans.lower().startswith("yes")


def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def clean_text(text):
    text = re.sub(r"\s+", " ", text).strip()
    cleaned = []
    for s in split_sentences(text):
        # شيل الجمل الذاتية أو الفاضية
        if re.search(r"\b(handsome|beautiful|attractive|stunning|gorgeous|cute|overall appearance|"
                     r"posing for|camera|appears to|seems to|well-defined|in the image|in the picture)\b",
                     s, flags=re.I):
            continue
        # شيل بادئة "The man is / The dog has / This chair is ..."
        s = re.sub(r"^(The|This|A|An)\s+(\w+\s+){0,2}?(is|has|are|have|wearing)\s+", "", s, flags=re.I)
        s = s.strip()
        if s:
            s = s[0].upper() + s[1:]
            cleaned.append(s)
    text = " ".join(cleaned)

    if text and text[-1] not in ".!":
        last_dot = text.rfind(".")
        text = text[:last_dot + 1] if last_dot > 40 else text + "."
    return text.strip()


def is_none_answer(text):
    t = text.lower().strip()
    return t.startswith("none") or t.startswith("no ") or "no accessories" in t or "not visible" in t


def remove_sentences(text, pattern):
    kept = [s for s in split_sentences(text) if not re.search(pattern, s, flags=re.I)]
    return " ".join(kept)


def detect_category(image):
    ans = ask_llava(
        image,
        "Is the main subject of this image a person, an animal, or an object? "
        "Answer with exactly one word: person, animal, or object.",
        max_new_tokens=4
    ).lower()
    if "person" in ans or "human" in ans or "man" in ans or "woman" in ans:
        return "person"
    if "animal" in ans:
        return "animal"
    return "object"


def verify_description(image, description, category):
    # 1) تحقق عام من الإكسسوارات اللي بيهلوس فيها النموذج
    for pattern, name in VERIFY_ITEMS.items():
        if re.search(pattern, description, flags=re.I):
            if not yes_no(image, f"Is {name} clearly visible on the main subject?"):
                description = remove_sentences(description, pattern)

    # 2) تحقق من الإيدين في الجيب (للأشخاص بس)
    if category == "person" and re.search(r"\bpockets?\b", description, flags=re.I):
        if not yes_no(image, "Are both hands inside pockets?"):
            description = remove_sentences(description, r"\bpockets?\b")
            if yes_no(image, "Is exactly one hand inside a pocket?"):
                description += " One hand is in a pocket."

    # 3) شيل الجمل المكررة
    seen, unique = set(), []
    for s in split_sentences(description):
        key = re.sub(r"\W+", " ", s.lower()).strip()
        if key and key not in seen:
            seen.add(key)
            unique.append(s)
    return " ".join(unique)

# ==========================================
# 4. الدالة الرئيسية: صورة -> Prompt قوي
# ==========================================
def get_3d_prompt_from_url(source, verbose=True):
    image = load_image(source)

    # المرحلة 1: النوع + اسم الـ subject
    category = detect_category(image)
    subject = ask_llava(
        image,
        "What is the single main subject in this image? "
        "Answer with a short noun phrase only (e.g. 'red leather sneaker', 'young man', 'golden retriever').",
        max_new_tokens=20
    ).rstrip(".")

    if verbose:
        print(f"[النوع: {category}] [الـ subject: {subject}]")

    # المرحلة 2: أسئلة مخصصة للنوع
    parts = []
    for q, tokens in QUESTION_SETS[category]:
        answer = ask_llava(image, q.format(s=subject), max_new_tokens=tokens)
        if category == "person" and "accessories" in q and is_none_answer(answer):
            continue
        answer = clean_text(answer)
        if answer:
            parts.append(answer)

    description = " ".join(parts)

    # المرحلة 3: تحقق وتنضيف
    description = verify_description(image, description, category)

    return f"{subject.capitalize()}. {description} {WHITE_BG_SUFFIX}"


# ==========================================
# 5. Inference Execution on Actual Cell 1 Image
# ==========================================
# Source input variable for LLaVA
image_url = STAGE1_IMAGE_PATH

prompt_result = get_3d_prompt_from_url(image_url)

if not prompt_result or not prompt_result.strip():
    raise RuntimeError("[Cell 2 Error] LLaVA prompt reasoning returned empty result during current execution.")

print("\n" + "=" * 60)
print("🎯 [Cell 2 Output] Actual Generated 3D-Aware Prompt:")
print(prompt_result)
print("=" * 60)

# Save actual prompt to STAGE2_PROMPT_PATH
generated_3d_prompt = prompt_result
with open(STAGE2_PROMPT_PATH, "w", encoding="utf-8") as f:
    f.write(generated_3d_prompt)

if not os.path.exists(STAGE2_PROMPT_PATH) or os.path.getsize(STAGE2_PROMPT_PATH) == 0:
    raise RuntimeError(f"[Cell 2 Error] Failed to verify newly saved prompt file at: {STAGE2_PROMPT_PATH}")

file_size = os.path.getsize(STAGE2_PROMPT_PATH)
print(f"\n✓ [Cell 2 Output] Successfully saved 3D prompt to: {STAGE2_PROMPT_PATH} ({file_size} bytes)")

# Record Cell 2 in ledger
try:
    with open(LEDGER_FILE, "r", encoding="utf-8") as f:
        ledger = json.load(f)
    ledger["stages"]["cell_2"] = {
        "status": "COMPLETED",
        "consumed_input_path": STAGE1_IMAGE_PATH,
        "source_var": "prompt_result",
        "dest_var": "prompt (Cell 3)",
        "output_path": STAGE2_PROMPT_PATH,
        "prompt_length": len(prompt_result),
        "file_size": file_size,
        "timestamp": os.path.getmtime(STAGE2_PROMPT_PATH)
    }
    with open(LEDGER_FILE, "w", encoding="utf-8") as f:
        json.dump(ledger, f, indent=2)
except Exception as e:
    print(f"[Cell 2 Warning] Could not update ledger: {e}")

# Clean up GPU VRAM for Stage 3
for v in ['model', 'processor', 'generate_ids', 'inputs']:
    if v in locals():
        try: del locals()[v]
        except Exception: pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"[Cell 2 VRAM] GPU memory freed. Allocated: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")

In [ ]:
# # =====================================================================
# # CELL 3 — PROMPT TO REFINED IMAGE: 2D Studio Refinement via Stable Diffusion 3.5
# # =====================================================================
# Consumes the actual prompt from Cell 2: /kaggle/working/brain3d_handoff/02_mllm_3d_prompt.txt
# Generates high-fidelity, studio-isolated 2D render with clean geometry.
# Saves actual refined image to: /kaggle/working/brain3d_handoff/03_diffusion_refined_image.png
# =====================================================================

import os
import sys
import gc
import json
import torch
from IPython.display import display

STAGE2_PROMPT_PATH = "/kaggle/working/brain3d_handoff/02_mllm_3d_prompt.txt"
STAGE3_REFINED_IMAGE_PATH = "/kaggle/working/brain3d_handoff/03_diffusion_refined_image.png"
LEDGER_FILE = "/kaggle/working/brain3d_handoff/execution_ledger.json"

# Purge any stale Stage 3 refined image artifact from previous runs
if os.path.exists(STAGE3_REFINED_IMAGE_PATH):
    try: os.remove(STAGE3_REFINED_IMAGE_PATH)
    except Exception: pass

# STRICT RULE: Must consume actual prompt from Cell 2 without fake or hardcoded fallbacks
if not os.path.exists(STAGE2_PROMPT_PATH) or os.path.getsize(STAGE2_PROMPT_PATH) == 0:
    raise FileNotFoundError(
        f"[Cell 3 Error] 3D prompt from Cell 2 not found or empty at: {STAGE2_PROMPT_PATH}.\n"
        "Cell 2 must complete successfully during current execution before running Cell 3."
    )

with open(STAGE2_PROMPT_PATH, "r", encoding="utf-8") as f:
    prompt = f.read().strip()

if not prompt:
    raise ValueError(f"[Cell 3 Error] Prompt file at {STAGE2_PROMPT_PATH} is empty.")

print(f"✓ [Cell 3 Input] Consuming actual prompt from Cell 2:\n{prompt}\n")

# --- Stage 3 Original Cell 0 ---
!nvidia-smi

# --- Stage 3 Original Cell 1 ---
!pip install -U diffusers transformers accelerate sentencepiece

# --- Stage 3 Original Cell 2 ---
from huggingface_hub import login

login()

# --- Stage 3 Original Cell 3 ---
import torch
from diffusers import StableDiffusion3Pipeline

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# --- Stage 3 Original Cell 4 ---

pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    torch_dtype=torch.float16
)

print("Model loaded successfully!")

# --- Stage 3 Original Cell 5 ---
pipe.enable_model_cpu_offload()

# =====================================================================
# Execute Refined 2D Synthesis using Actual Cell 2 Prompt
# =====================================================================
print("\n--- Generating Refined 2D Image with Actual Stage 2 Prompt ---")
# Source inference call using actual ingested prompt
image = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=4.5
).images[0]

if image is None:
    raise RuntimeError("[Cell 3 Error] Stable Diffusion pipeline produced no image (image is None).")

display(image)

# Save actual refined image for Stage 4 TRELLIS
image.save(STAGE3_REFINED_IMAGE_PATH)
if not os.path.exists(STAGE3_REFINED_IMAGE_PATH) or os.path.getsize(STAGE3_REFINED_IMAGE_PATH) == 0:
    raise RuntimeError(f"[Cell 3 Error] Failed to verify newly saved refined image at: {STAGE3_REFINED_IMAGE_PATH}")

file_size = os.path.getsize(STAGE3_REFINED_IMAGE_PATH)
dims = list(image.size)
print(f"\n✓ [Cell 3 Output] Successfully saved refined image to: {STAGE3_REFINED_IMAGE_PATH} ({file_size/1024:.1f} KB, dimensions: {dims})")

# Record Cell 3 in ledger
try:
    with open(LEDGER_FILE, "r", encoding="utf-8") as f:
        ledger = json.load(f)
    ledger["stages"]["cell_3"] = {
        "status": "COMPLETED",
        "consumed_input_path": STAGE2_PROMPT_PATH,
        "source_var": "image",
        "dest_var": "IMAGE_PATH (Cell 4 / TRELLIS subprocess)",
        "output_path": STAGE3_REFINED_IMAGE_PATH,
        "dimensions": dims,
        "file_size": file_size,
        "timestamp": os.path.getmtime(STAGE3_REFINED_IMAGE_PATH)
    }
    with open(LEDGER_FILE, "w", encoding="utf-8") as f:
        json.dump(ledger, f, indent=2)
except Exception as e:
    print(f"[Cell 3 Warning] Could not update ledger: {e}")

# Clean up GPU VRAM for Stage 4
for v in ['pipe', 'image']:
    if v in locals():
        try: del locals()[v]
        except Exception: pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(f"[Cell 3 VRAM] GPU memory freed. Allocated: {torch.cuda.memory_allocated() / (1024**2):.1f} MB")

In [ ]:
# # =====================================================================
# # CELL 4 — REFINED IMAGE TO 3D: High-Fidelity 3D Reconstruction via TRELLIS
# # =====================================================================
# Consumes the actual refined image from Cell 3: /kaggle/working/brain3d_handoff/03_diffusion_refined_image.png
# Runs complete TRELLIS setup (venv, CUDA 11.8, custom extensions) and inference pipeline.
# Generates Radiance Fields, 3D Gaussians, and extracts Textured Mesh.
# Exports final GLB to: /kaggle/working/trellis_output/trellis_result.glb
# =====================================================================

import os
import sys
import subprocess
import shutil
import time
import json
import textwrap

STAGE3_REFINED_IMAGE_PATH = "/kaggle/working/brain3d_handoff/03_diffusion_refined_image.png"
STAGE4_GLB_OUTPUT_DIR = "/kaggle/working/trellis_output"
os.makedirs(STAGE4_GLB_OUTPUT_DIR, exist_ok=True)
STAGE4_GLB_PATH = "/kaggle/working/trellis_output/trellis_result.glb"
STAGE4_VIEWER_PATH = "/kaggle/working/trellis_output/trellis_viewer.html"
GLB_PATH = STAGE4_GLB_PATH
HTML_PATH = STAGE4_VIEWER_PATH
OUT_GLB = STAGE4_GLB_PATH
LEDGER_FILE = "/kaggle/working/brain3d_handoff/execution_ledger.json"

# Explicitly export environment variable for subprocesses
os.environ["TRELLIS_IMAGE_PATH"] = STAGE3_REFINED_IMAGE_PATH

# Purge any stale Stage 4 GLB and HTML artifacts from previous runs
if os.path.exists(GLB_PATH):
    try: os.remove(GLB_PATH)
    except Exception: pass
if os.path.exists(HTML_PATH):
    try: os.remove(HTML_PATH)
    except Exception: pass

# STRICT RULE: Must consume actual refined image from Cell 3 without fake fallbacks
if not os.path.exists(STAGE3_REFINED_IMAGE_PATH) or os.path.getsize(STAGE3_REFINED_IMAGE_PATH) == 0:
    raise FileNotFoundError(
        f"[Cell 4 Error] Refined image from Cell 3 not found or empty at: {STAGE3_REFINED_IMAGE_PATH}.\n"
        "Cell 3 must complete successfully during current execution before running Cell 4."
    )
print(f"✓ [Cell 4 Input] Verified actual refined image from Cell 3: {STAGE3_REFINED_IMAGE_PATH} ({os.path.getsize(STAGE3_REFINED_IMAGE_PATH)/1024:.1f} KB)")

# --- Stage 4 TRELLIS Setup (Cell 1 of 2d_3d.ipynb) ---
# =====================================================================
# CELL 1 — COMPLETE TRELLIS SETUP  (idempotent; safe to re-run)
# Requirements: Kaggle Internet ON, GPU T4 x2 (or any T4)
# =====================================================================
import os, sys, subprocess, shutil, time, textwrap

ROOT  = "/kaggle/working"
VENV  = f"{ROOT}/trellis_env"
PY    = f"{VENV}/bin/python"
REPO  = f"{ROOT}/TRELLIS"
EXT   = f"{ROOT}/extensions"
CUDA  = "/usr/local/cuda-11.8"
HF    = f"{ROOT}/hf_cache"
CONSTRAINTS = f"{ROOT}/constraints.txt"
os.makedirs(EXT, exist_ok=True)

# ---- environment used for every subprocess (build + runtime) ----
ENV = os.environ.copy()
ENV.update({
    "CUDA_HOME": CUDA,
    "CUDA_PATH": CUDA,
    "PATH": f"{VENV}/bin:{CUDA}/bin:" + ENV.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA}/lib64:" + ENV.get("LD_LIBRARY_PATH", ""),
    "TORCH_CUDA_ARCH_LIST": "7.5",          # Tesla T4
    "MAX_JOBS": "4",
    "FORCE_CUDA": "1",
    "NVCC_APPEND_FLAGS": "-allow-unsupported-compiler",
    "ATTN_BACKEND": "xformers",
    "SPCONV_ALGO": "native",
    "HF_HOME": HF,
    "PYTHONPATH": REPO,
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
})

def tail(s, n=150):
    lines = (s or "").strip().splitlines()
    return "\n".join(lines[-n:])

def sh(cmd, desc="", cwd=None, check=True):
    """Run a command; on failure print the REAL stdout/stderr and raise."""
    shell = isinstance(cmd, str)
    t0 = time.time()
    if desc: print(f"\n>>> {desc}", flush=True)
    p = subprocess.run(cmd, shell=shell, cwd=cwd, env=ENV,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0 and check:
        print("\n" + "!" * 78)
        print(f"COMMAND FAILED (exit {p.returncode}): {cmd if shell else ' '.join(cmd)}")
        print("----- STDOUT (tail) -----\n" + tail(p.stdout))
        print("----- STDERR (tail) -----\n" + tail(p.stderr))
        print("!" * 78)
        raise RuntimeError(f"Step failed: {desc or cmd}")
    if desc: print(f"    done in {time.time()-t0:.0f}s", flush=True)
    return p

def py_ok(code):
    """True if `code` runs without error inside the TRELLIS venv."""
    return subprocess.run([PY, "-c", code], env=ENV, capture_output=True, text=True).returncode == 0

def uv_install(args, desc, extra=None):
    cmd = ["uv", "pip", "install", "--python", PY, "-c", CONSTRAINTS,
           "--index-strategy", "unsafe-best-match"] + args + (extra or [])
    sh(cmd, desc)

print("=" * 78, "\nTRELLIS SETUP (Python 3.10 / torch 2.4.0+cu118 / T4)\n" + "=" * 78)

# ---------------------------------------------------------------- 1. CUDA 11.8
if os.path.exists(f"{CUDA}/bin/nvcc"):
    print(f"[1/12] CUDA 11.8 toolkit already present: {CUDA}")
else:
    print("[1/12] Installing CUDA Toolkit 11.8 (toolkit only, ~4 GB download)...")
    run_file = "/tmp/cuda_11.8.0_520.61.05_linux.run"
    if not os.path.exists(run_file):
        sh(["wget", "-q", "-c", "-O", run_file,
            "https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run"],
           "Downloading CUDA 11.8 installer")
    sh(["sh", run_file, "--silent", "--toolkit", f"--toolkitpath={CUDA}",
        "--no-opengl-libs", "--no-drm", "--no-man-page", "--override"],
       "Installing CUDA 11.8 toolkit")
    if os.path.exists(run_file): os.remove(run_file)
assert os.path.exists(f"{CUDA}/bin/nvcc"), "nvcc 11.8 not found after install"
print(sh([f"{CUDA}/bin/nvcc", "--version"]).stdout.strip().splitlines()[-1])

# ---------------------------------------------------------------- 2. Python 3.10 venv (uv)
if os.path.exists(PY) and py_ok("import sys; assert sys.version_info[:2]==(3,10)"):
    print("[2/12] Python 3.10 venv already exists")
else:
    print("[2/12] Creating Python 3.10 venv with uv (no ensurepip)...")
    sh([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], "Installing uv")
    if os.path.exists(VENV): shutil.rmtree(VENV)
    sh(["uv", "venv", "--python", "3.10", "--seed", VENV], "Creating venv (Python 3.10)")
assert os.path.exists(PY)
print(sh([PY, "--version"]).stdout.strip())

# constraints: nothing later may change these
with open(CONSTRAINTS, "w") as f:
    f.write("torch==2.4.0\ntorchvision==0.19.0\nnumpy<2\ntransformers==4.50.1\n")

# ---------------------------------------------------------------- 3. Build tooling
if py_ok("import setuptools, wheel, ninja"):
    print("[3/12] Build tooling present")
else:
    uv_install(["setuptools<70", "wheel", "ninja", "packaging", "numpy<2", "cmake"],
               "[3/12] Installing build tooling")

# ---------------------------------------------------------------- 4. PyTorch
if py_ok("import torch,torchvision; assert torch.__version__.startswith('2.4.0') "
         "and torch.version.cuda=='11.8' and torchvision.__version__.startswith('0.19.0')"):
    print("[4/12] torch 2.4.0+cu118 / torchvision 0.19.0 already installed")
else:
    uv_install(["torch==2.4.0+cu118", "torchvision==0.19.0+cu118"],
               "[4/12] Installing PyTorch 2.4.0 + torchvision 0.19.0 (cu118)",
               extra=["--index-url", "https://download.pytorch.org/whl/cu118",
                      "--extra-index-url", "https://pypi.org/simple"])

# ---------------------------------------------------------------- 5. TRELLIS repo
if os.path.exists(f"{REPO}/trellis/__init__.py"):
    print("[5/12] TRELLIS repo already cloned")
    sh(["git", "submodule", "update", "--init", "--recursive"], "Syncing submodules", cwd=REPO, check=False)
else:
    if os.path.exists(REPO): shutil.rmtree(REPO)
    sh(["git", "clone", "--recurse-submodules", "https://github.com/microsoft/TRELLIS.git", REPO],
       "[5/12] Cloning official TRELLIS (with submodules)")

# ---------------------------------------------------------------- 6. Basic deps + transformers
if py_ok("import plyfile, trimesh, utils3d, rembg, onnxruntime, open3d, xatlas, pyvista, pymeshfix, igraph, "
         "imageio, easydict, cv2, scipy, transformers; assert transformers.__version__=='4.50.1'"):
    print("[6/12] TRELLIS basic deps already installed")
else:
    uv_install(["pillow", "imageio", "imageio-ffmpeg", "tqdm", "easydict", "opencv-python-headless",
                "scipy", "ninja", "rembg", "onnxruntime", "trimesh", "open3d", "xatlas", "pyvista",
                "pymeshfix", "igraph", "plyfile", "einops", "psutil", "safetensors", "huggingface_hub",
                "transformers==4.50.1",
                "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"],
               "[6/12] Installing TRELLIS basic dependencies + transformers 4.50.1 + plyfile + trimesh + utils3d")

# ---------------------------------------------------------------- 7. xformers
if py_ok("import xformers; assert xformers.__version__.startswith('0.0.27.post2')"):
    print("[7/12] xformers 0.0.27.post2 already installed")
else:
    uv_install(["xformers==0.0.27.post2", "--no-deps"],
               "[7/12] Installing xformers 0.0.27.post2 (cu118)",
               extra=["--index-url", "https://download.pytorch.org/whl/cu118"])

# ---------------------------------------------------------------- 8. spconv
if py_ok("import spconv.pytorch"):
    print("[8/12] spconv already installed")
else:
    uv_install(["spconv-cu118"], "[8/12] Installing spconv-cu118")

# ---------------------------------------------------------------- 9. nvdiffrast
if py_ok("import nvdiffrast.torch"):
    print("[9/12] nvdiffrast already installed")
else:
    d = f"{EXT}/nvdiffrast"
    if not os.path.exists(d):
        sh(["git", "clone", "https://github.com/NVlabs/nvdiffrast.git", d], "Cloning nvdiffrast")
    uv_install([d, "--no-build-isolation"], "[9/12] Installing nvdiffrast")

# ---------------------------------------------------------------- 10. Kaolin 0.16.0
if py_ok("import kaolin; assert kaolin.__version__=='0.16.0'"):
    print("[10/12] Kaolin 0.16.0 already installed")
else:
    uv_install(["kaolin==0.16.0"], "[10/12] Installing Kaolin 0.16.0 (torch-2.4.0_cu118)",
               extra=["-f", "https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu118.html"])

# ---------------------------------------------------------------- 11. diffoctreerast + mip-splatting
if py_ok("import diffoctreerast"):
    print("[11a/12] diffoctreerast already installed")
else:
    d = f"{EXT}/diffoctreerast"
    if not os.path.exists(d):
        sh(["git", "clone", "--recurse-submodules", "https://github.com/JeffreyXiang/diffoctreerast.git", d],
           "Cloning diffoctreerast")
    uv_install([d, "--no-build-isolation"], "[11a/12] Compiling diffoctreerast (CUDA, a few minutes)")

if py_ok("import diff_gaussian_rasterization"):
    print("[11b/12] diff-gaussian-rasterization already installed")
else:
    d = f"{EXT}/mip-splatting"
    if not os.path.exists(d):
        sh(["git", "clone", "--recursive", "https://github.com/autonomousvision/mip-splatting.git", d],
           "Cloning mip-splatting (recursive, includes glm)")
    sub = f"{d}/submodules/diff-gaussian-rasterization"
    sh(["git", "submodule", "update", "--init", "--recursive"], "Syncing rasterizer submodules", cwd=sub, check=False)
    uv_install([sub, "--no-build-isolation"], "[11b/12] Compiling diff-gaussian-rasterization (CUDA, a few minutes)")

# ---------------------------------------------------------------- 12. Verify + import TRELLIS
print("\n[12/12] Verifying environment...")
verify = textwrap.dedent(r'''
import os, importlib
import torch, torchvision, transformers, xformers, trimesh
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("XFormers:", xformers.__version__)
import spconv.pytorch;            print("spconv: OK")
import nvdiffrast.torch;          print("nvdiffrast: OK")
import kaolin;                    print("Kaolin:", kaolin.__version__)
print("Trimesh:", trimesh.__version__)
import diffoctreerast;            print("diffoctreerast: OK")
import diff_gaussian_rasterization; print("diff-gaussian-rasterization: OK")
import plyfile, utils3d;          print("plyfile / utils3d: OK")
assert torch.__version__.startswith("2.4.0") and torch.version.cuda == "11.8" and torch.cuda.is_available()
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils
print("ATTN_BACKEND =", os.environ.get("ATTN_BACKEND"), "| SPCONV_ALGO =", os.environ.get("SPCONV_ALGO"))
print("TRELLIS import: OK")
''')
p = subprocess.run([PY, "-c", verify], env=ENV, capture_output=True, text=True)
print(p.stdout)
if p.returncode != 0:
    print("!" * 78, "\nVERIFICATION FAILED\n----- STDERR -----\n" + tail(p.stderr, 200), "\n" + "!" * 78)
    raise RuntimeError("TRELLIS verification failed (see stderr above)")
print("SETUP COMPLETE")

# --- Stage 4 TRELLIS Inference & GLB Export (Cell 2 of 2d_3d.ipynb) ---
# =====================================================================
# CELL 2 — IMAGE → TRELLIS-image-large → MESH → GLB  (low-disk version)
# Cell 1 is NOT modified. No script file is written; code runs via `python -c`.
# Model files are read from /kaggle/working/hf_cache when present; anything
# missing is downloaded to /tmp (NOT to /kaggle/working).
# =====================================================================
import os, sys, shutil, subprocess, glob

ROOT   = "/kaggle/working"
VENV   = f"{ROOT}/trellis_env"          # underscore = the ONE used by Cell 1 (kept)
DUPENV = f"{ROOT}/trellis-env"          # hyphen     = stale duplicate (removed if safe)
PYTHON = f"{VENV}/bin/python"
REPO   = f"{ROOT}/TRELLIS"
CUDA   = "/usr/local/cuda-11.8"
OUT_DIR = f"{ROOT}/trellis_output"

BAKE_TEXTURED_GLB = False      # optional 1024 texture (needs gaussian decoder; off by default)
MIN_FREE_GB       = 0.3

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def free_gb(path=ROOT):
    return shutil.disk_usage(path).free / 1024**3

def rm(path):
    if os.path.isdir(path) and not os.path.islink(path):
        shutil.rmtree(path, ignore_errors=True)
    elif os.path.lexists(path):
        try: os.remove(path)
        except OSError: pass

if not os.path.exists(PYTHON) or not os.path.exists(f"{REPO}/trellis"):
    raise RuntimeError("TRELLIS env/repo not found. Cell 1 must have been run in this session.")

# ---------------------------------------------------------------- 1. disk status + safe cleanup
print("=== DISK /kaggle/working (before) ===")
print(sh(f"df -h {ROOT}").stdout)

# (a) leftovers from previous failed/test runs (regenerated by this cell)
rm(f"{ROOT}/run_image_to_3d.py")
rm(OUT_DIR)
rm(f"{ROOT}/extensions")             # build sources (compiled packages already installed)
for p in ["/root/.cache/uv", "/root/.cache/pip", "/tmp/uv-cache"]:
    rm(p)
for g in glob.glob(f"{ROOT}/hf_cache/**/*.incomplete", recursive=True):
    rm(g)
for g in glob.glob(f"{REPO}/**/__pycache__", recursive=True):
    rm(g)

# (b) the duplicate 8.5 GB venv "trellis-env" (hyphen) -- only if trellis_env is verified healthy
if os.path.isdir(DUPENV) and os.path.realpath(DUPENV) != os.path.realpath(VENV):
    dup_real = os.path.realpath(DUPENV)
    py_real  = os.path.realpath(PYTHON)
    cfg_txt  = ""
    try:
        cfg_txt = open(f"{VENV}/pyvenv.cfg").read()
    except OSError:
        pass
    independent = (not py_real.startswith(dup_real + os.sep)) and ("trellis-env" not in cfg_txt)

    env_chk = os.environ.copy()
    env_chk.update({"CUDA_HOME": CUDA, "CUDA_VISIBLE_DEVICES": "0",
                    "LD_LIBRARY_PATH": f"{CUDA}/lib64:" + env_chk.get("LD_LIBRARY_PATH", "")})
    chk = subprocess.run(
        [PYTHON, "-c",
         "import torch, xformers, spconv.pytorch, nvdiffrast.torch, diffoctreerast, diff_gaussian_rasterization; "
         "print('ok')"],
        env=env_chk, capture_output=True, text=True)

    if independent and chk.returncode == 0:
        rm(DUPENV)
        print("Removed duplicate venv /kaggle/working/trellis-env (trellis_env verified healthy and independent).")
    else:
        print("KEPT trellis-env: safety check did not pass.")
        print("  independent of trellis_env:", independent)
        print("  trellis_env import check  :", "ok" if chk.returncode == 0 else chk.stderr[-400:])

print("\n=== DISK /kaggle/working (after) ===")
print(sh(f"df -h {ROOT}").stdout)
print("Largest items:")
print(sh(f"du -sh {ROOT}/* {ROOT}/.[!.]* 2>/dev/null | sort -h | tail -6").stdout)
print(f"Free in /kaggle/working: {free_gb():.2f} GB")
print(f"Free in /tmp (used for missing model files): {free_gb('/tmp'):.1f} GB")

if free_gb() < MIN_FREE_GB:
    raise RuntimeError(f"Only {free_gb():.2f} GB free in /kaggle/working - not enough to save the GLB. "
                       "See the listing above (do not delete TRELLIS, trellis_env or hf_cache).")
if free_gb("/tmp") < 6:
    raise RuntimeError("Less than 6 GB free in /tmp - cannot stage missing model files.")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------- 2. inference code (passed via -c, no file on disk)
CODE = r'''
import os, sys, json, gc
os.environ["CUDA_HOME"] = "/usr/local/cuda-11.8"
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.insert(0, "/kaggle/working/TRELLIS")

import numpy as np, torch, trimesh
from PIL import Image
from huggingface_hub import hf_hub_download
from trellis.pipelines import TrellisImageTo3DPipeline

IMAGE_PATH = os.environ.get("TRELLIS_IMAGE_PATH", "/kaggle/working/brain3d_handoff/03_diffusion_refined_image.png")
OUT_GLB    = "/kaggle/working/trellis_output/trellis_result.glb"
OUT_TEX    = "/kaggle/working/trellis_output/trellis_result_textured_1024.glb"
BAKE       = os.environ.get("BAKE_TEXTURED_GLB", "0") == "1"

REPO_ID    = "microsoft/TRELLIS-image-large"
WORK_CACHE = "/kaggle/working/hf_cache/hub"     # existing cache (read-only use)
TMP_CACHE  = "/tmp/hf_dl"                       # downloads for missing files go here
LOCAL      = "/tmp/trellis_local"               # small local pipeline folder (symlinks)

# ---- 1. image ----
image = Image.open(IMAGE_PATH).convert("RGB")
print("Image loaded")
print(f"Input image : {IMAGE_PATH}")
print(f"Image size  : {image.size[0]} x {image.size[1]}", flush=True)

# ---- 2. build a local pipeline folder: reuse cache, download only what is missing (to /tmp) ----
def fetch(fn):
    try:
        return hf_hub_download(REPO_ID, fn, cache_dir=WORK_CACHE, local_files_only=True), "cache"
    except Exception:
        return hf_hub_download(REPO_ID, fn, cache_dir=TMP_CACHE), "downloaded to /tmp"

cfg_path, _ = fetch("pipeline.json")
cfg = json.load(open(cfg_path))
skip = {"slat_decoder_rf"} | (set() if BAKE else {"slat_decoder_gs"})   # not needed for mesh output
models = {k: v for k, v in cfg["args"]["models"].items() if k not in skip}
cfg["args"]["models"] = models
print("Skipping decoders:", sorted(skip), flush=True)

os.makedirs(f"{LOCAL}/ckpts", exist_ok=True)
json.dump(cfg, open(f"{LOCAL}/pipeline.json", "w"))
for name, rel in models.items():
    for ext in (".json", ".safetensors"):
        src, how = fetch(rel + ext)
        dst = f"{LOCAL}/{rel}{ext}"
        if os.path.lexists(dst):
            os.remove(dst)
        os.symlink(os.path.realpath(src), dst)
        if ext == ".safetensors":
            print(f"  {name:28s} <- {how}", flush=True)

# ---- 3. pipeline ----
print("GPU         :", torch.cuda.get_device_name(0), flush=True)
pipeline = TrellisImageTo3DPipeline.from_pretrained(LOCAL)
pipeline.cuda()
torch.cuda.empty_cache()
print("TRELLIS pipeline loaded")
print(f"GPU mem after load: {torch.cuda.memory_allocated()/1024**3:.2f} GB", flush=True)

# ---- 4. image -> 3D ----
formats = ["mesh", "gaussian"] if BAKE else ["mesh"]
outputs = pipeline.run(
    image,
    seed=42,
    formats=formats,
    sparse_structure_sampler_params={"steps": 12, "cfg_strength": 7.5},
    slat_sampler_params={"steps": 12, "cfg_strength": 3.0},
)
mesh = outputs["mesh"][0]
print("3D mesh generated", flush=True)

# ---- 5. export raw GLB ----
verts = mesh.vertices.detach().float().cpu().numpy()
faces = mesh.faces.detach().cpu().numpy()
n_v, n_f = len(verts), len(faces)
verts = verts @ np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], dtype=np.float32)   # Z-up -> Y-up (glTF)

colors = None
va = getattr(mesh, "vertex_attrs", None)
if va is not None:
    va = va.detach().float().cpu().numpy()
    if va.ndim == 2 and va.shape[0] == n_v and va.shape[1] >= 3:
        colors = (np.clip(va[:, :3], 0, 1) * 255).astype(np.uint8)
    del va

trimesh.Trimesh(vertices=verts, faces=faces, vertex_colors=colors, process=False).export(OUT_GLB)
del verts, faces, colors
print("GLB exported successfully", flush=True)

print("\n===== RESULT =====")
print(f"Vertices   : {n_v:,}")
print(f"Faces      : {n_f:,}")
print(f"Output     : {OUT_GLB}")
print(f"GLB size   : {os.path.getsize(OUT_GLB)/1024**2:.1f} MB")
print(f"GPU alloc  : {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU peak   : {torch.cuda.max_memory_allocated()/1024**3:.2f} GB", flush=True)

# ---- 6. optional textured GLB (disabled by default) ----
if BAKE:
    try:
        from trellis.utils import postprocessing_utils
        torch.cuda.empty_cache(); gc.collect()
        glb = postprocessing_utils.to_glb(outputs["gaussian"][0], mesh, simplify=0.95, texture_size=1024)
        glb.export(OUT_TEX)
        print(f"Textured GLB: {OUT_TEX} ({os.path.getsize(OUT_TEX)/1024**2:.1f} MB)")
    except Exception as e:
        print("Textured GLB skipped (non-fatal):", repr(e))
'''

# ---------------------------------------------------------------- 3. run inside the TRELLIS venv
env = os.environ.copy()
env.update({
    "TRELLIS_IMAGE_PATH": STAGE3_REFINED_IMAGE_PATH,
    "CUDA_HOME": CUDA,
    "PATH": f"{VENV}/bin:{CUDA}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA}/lib64:" + env.get("LD_LIBRARY_PATH", ""),
    "ATTN_BACKEND": "xformers",
    "SPCONV_ALGO": "native",
    "HF_HOME": f"{ROOT}/hf_cache",
    "PYTHONPATH": REPO,
    "CUDA_VISIBLE_DEVICES": "0",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TORCH_CUDA_ARCH_LIST": "7.5",
    "PYTHONUNBUFFERED": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "BAKE_TEXTURED_GLB": "1" if BAKE_TEXTURED_GLB else "0",
})

proc = subprocess.Popen([PYTHON, "-u", "-c", CODE], env=env, cwd=REPO,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Image→3D failed with exit code {rc} (full traceback printed above)")

print(f"\nFree disk after run: {free_gb():.2f} GB")

# =====================================================================
# Verify Output GLB Mesh Generation
# =====================================================================
if not os.path.exists(GLB_PATH) or os.path.getsize(GLB_PATH) == 0:
    raise RuntimeError(
        f"[Cell 4 Error] TRELLIS inference completed but did not produce a valid GLB at: {GLB_PATH}."
    )

file_size = os.path.getsize(GLB_PATH)
print(f"\n✓ [Cell 4 Output] Successfully verified 3D GLB mesh: {GLB_PATH} ({file_size/1024:.1f} KB)")

# Record Cell 4 in ledger
try:
    with open(LEDGER_FILE, "r", encoding="utf-8") as f:
        ledger = json.load(f)
    ledger["stages"]["cell_4"] = {
        "status": "COMPLETED",
        "consumed_input_path": STAGE3_REFINED_IMAGE_PATH,
        "source_var": "glb (inside TRELLIS subprocess)",
        "dest_var": "GLB_PATH (Cell 5 / 3D Viewer)",
        "output_path": GLB_PATH,
        "file_size": file_size,
        "timestamp": os.path.getmtime(GLB_PATH)
    }
    with open(LEDGER_FILE, "w", encoding="utf-8") as f:
        json.dump(ledger, f, indent=2)
except Exception as e:
    print(f"[Cell 4 Warning] Could not update ledger: {e}")

In [ ]:
# # =====================================================================
# # CELL 5 — FINAL VALIDATION AND VIEWER: Automated Data-Flow Audit & 3D WebGL Viewer
# # =====================================================================
# Verifies real outputs of previous 4 cells (no stale or fake artifacts allowed).
# Evaluates each stage of the data flow and reports [PASS/FAIL/NOT RUN].
# Loads the final GLB and displays the interactive 3D WebGL mesh viewer.
# =====================================================================

import os
import sys
import time
import json
import datetime
from IPython.display import display, Image as IPImage

# Canonical artifact paths
LEDGER_FILE               = "/kaggle/working/brain3d_handoff/execution_ledger.json"
STAGE1_IMAGE_PATH         = "/kaggle/working/brain3d_handoff/01_eeg_decoded_image.png"
STAGE2_PROMPT_PATH        = "/kaggle/working/brain3d_handoff/02_mllm_3d_prompt.txt"
STAGE3_REFINED_IMAGE_PATH = "/kaggle/working/brain3d_handoff/03_diffusion_refined_image.png"
STAGE4_GLB_PATH           = "/kaggle/working/trellis_output/trellis_result.glb"
STAGE4_VIEWER_PATH        = "/kaggle/working/trellis_output/trellis_viewer.html"
GLB_PATH                  = STAGE4_GLB_PATH
HTML_PATH                 = STAGE4_VIEWER_PATH

# Load execution ledger
ledger = None
current_run_start = None
run_id = "UNKNOWN"
if os.path.exists(LEDGER_FILE):
    try:
        with open(LEDGER_FILE, "r", encoding="utf-8") as f:
            ledger = json.load(f)
            current_run_start = ledger.get("run_start_time")
            run_id = ledger.get("run_id", "UNKNOWN")
    except Exception:
        pass

print("=" * 85)
print("BRAIN3D PIPELINE: AUTOMATED END-TO-END DATA-FLOW AUDIT")
if current_run_start is not None:
    start_dt = datetime.datetime.fromtimestamp(current_run_start).strftime('%Y-%m-%d %H:%M:%S')
    print(f"Run ID: {run_id} | Execution Started: {start_dt}")
else:
    print("Run Status: No active execution session detected (Cell 1 has not run in this session)")
print("=" * 85)

# -------------------------------------------------------------------------
# 1. EVALUATE THE 8 MANDATORY AUDIT CRITERIA
# -------------------------------------------------------------------------
stages_data = ledger.get("stages", {}) if ledger else {}

# Criterion 1: EEG inference generated current run's image
c1_exists = os.path.exists(STAGE1_IMAGE_PATH) and os.path.getsize(STAGE1_IMAGE_PATH) > 0
c1_fresh = c1_exists and (current_run_start is not None) and (os.path.getmtime(STAGE1_IMAGE_PATH) >= current_run_start - 2.0)
c1_logged = "cell_1" in stages_data and stages_data["cell_1"].get("status") == "COMPLETED"

if c1_fresh and c1_logged:
    crit1 = "PASS"
elif not c1_exists and not c1_logged:
    crit1 = "NOT RUN"
else:
    crit1 = "FAIL"

# Criterion 2: LLaVA consumed that exact image
c2_logged = "cell_2" in stages_data and stages_data["cell_2"].get("status") == "COMPLETED"
if crit1 == "PASS" and c2_logged and stages_data["cell_2"].get("consumed_input_path") == STAGE1_IMAGE_PATH:
    crit2 = "PASS"
elif "cell_2" not in stages_data:
    crit2 = "NOT RUN"
else:
    crit2 = "FAIL"

# Criterion 3: LLaVA generated a non-empty prompt
c2_exists = os.path.exists(STAGE2_PROMPT_PATH) and os.path.getsize(STAGE2_PROMPT_PATH) > 0
c2_fresh = c2_exists and (current_run_start is not None) and (os.path.getmtime(STAGE2_PROMPT_PATH) >= current_run_start - 2.0)
if c2_fresh and c2_logged:
    crit3 = "PASS"
elif not c2_exists and not c2_logged:
    crit3 = "NOT RUN"
else:
    crit3 = "FAIL"

# Criterion 4: Stable Diffusion consumed that exact prompt
c3_logged = "cell_3" in stages_data and stages_data["cell_3"].get("status") == "COMPLETED"
if crit3 == "PASS" and c3_logged and stages_data["cell_3"].get("consumed_input_path") == STAGE2_PROMPT_PATH:
    crit4 = "PASS"
elif "cell_3" not in stages_data:
    crit4 = "NOT RUN"
else:
    crit4 = "FAIL"

# Criterion 5: Stable Diffusion generated the current run's image
c3_exists = os.path.exists(STAGE3_REFINED_IMAGE_PATH) and os.path.getsize(STAGE3_REFINED_IMAGE_PATH) > 0
c3_fresh = c3_exists and (current_run_start is not None) and (os.path.getmtime(STAGE3_REFINED_IMAGE_PATH) >= current_run_start - 2.0)
if c3_fresh and c3_logged:
    crit5 = "PASS"
elif not c3_exists and not c3_logged:
    crit5 = "NOT RUN"
else:
    crit5 = "FAIL"

# Criterion 6: TRELLIS consumed that exact refined image
c4_logged = "cell_4" in stages_data and stages_data["cell_4"].get("status") == "COMPLETED"
if crit5 == "PASS" and c4_logged and stages_data["cell_4"].get("consumed_input_path") == STAGE3_REFINED_IMAGE_PATH:
    crit6 = "PASS"
elif "cell_4" not in stages_data:
    crit6 = "NOT RUN"
else:
    crit6 = "FAIL"

# Criterion 7: TRELLIS generated the current run's GLB
c4_exists = os.path.exists(GLB_PATH) and os.path.getsize(GLB_PATH) > 0
c4_fresh = c4_exists and (current_run_start is not None) and (os.path.getmtime(GLB_PATH) >= current_run_start - 2.0)
if c4_fresh and c4_logged:
    crit7 = "PASS"
elif not c4_exists and not c4_logged:
    crit7 = "NOT RUN"
else:
    crit7 = "FAIL"

# Criterion 8: The viewer loaded the current run's GLB
crit8 = "PASS" if crit7 == "PASS" else ("NOT RUN" if crit7 == "NOT RUN" else "FAIL")

audit_checklist = [
    (crit1, "EEG inference generated the current run's image."),
    (crit2, "LLaVA consumed that exact image."),
    (crit3, "LLaVA generated a non-empty prompt."),
    (crit4, "Stable Diffusion consumed that exact prompt."),
    (crit5, "Stable Diffusion generated the current run's image."),
    (crit6, "TRELLIS consumed that exact refined image."),
    (crit7, "TRELLIS generated the current run's GLB."),
    (crit8, "The viewer loaded the current run's GLB.")
]

print("\n--- Mandatory Automated Verification Checklist ---")
for status, desc in audit_checklist:
    print(f"[{status:7s}] {desc}")

# -------------------------------------------------------------------------
# 2. RESOLVED ARTIFACT PATHS, SIZES, DIMENSIONS & VARIABLES
# -------------------------------------------------------------------------
print("\n" + "-" * 85)
print(f"{'Stage':26s} | {'Source Var':18s} -> {'Dest Var':18s} | Details / Dimensions")
print("-" * 85)

handoff_records = [
    (
        "Cell 1: EEG-to-Image",
        "sample_img",
        "image_url",
        STAGE1_IMAGE_PATH,
        stages_data.get("cell_1", {}).get("dimensions", ["Unknown"]),
        crit1
    ),
    (
        "Cell 2: LLaVA Reasoning",
        "prompt_result",
        "prompt",
        STAGE2_PROMPT_PATH,
        f"{stages_data.get('cell_2', {}).get('prompt_length', 0)} chars",
        crit3
    ),
    (
        "Cell 3: Diffusion Refine",
        "image",
        "IMAGE_PATH",
        STAGE3_REFINED_IMAGE_PATH,
        stages_data.get("cell_3", {}).get("dimensions", ["Unknown"]),
        crit5
    ),
    (
        "Cell 4: TRELLIS 3D",
        "glb (subprocess)",
        "GLB_PATH",
        GLB_PATH,
        "3D Mesh (.glb)",
        crit7
    )
]

for stage_name, src_var, dst_var, fpath, extra_info, status in handoff_records:
    size_str = f"{os.path.getsize(fpath)/1024:.1f} KB" if os.path.exists(fpath) else "MISSING"
    print(f"{stage_name:26s} | {src_var:18s} -> {dst_var:18s} | {size_str} ({extra_info})")
    print(f"  └─ Path: {fpath} [{status}]")

print("-" * 85)

# Display intermediate visual and textual outputs if valid for current execution
if crit1 == "PASS":
    print("\n--- Cell 1 Current Output: Decoded EEG Image ---")
    display(IPImage(filename=STAGE1_IMAGE_PATH))

if crit3 == "PASS":
    print("\n--- Cell 2 Current Output: 3D-Aware Semantic Prompt ---")
    with open(STAGE2_PROMPT_PATH, "r", encoding="utf-8") as f:
        print(f.read().strip())

if crit5 == "PASS":
    print("\n--- Cell 3 Current Output: Refined 2D Diffusion Image ---")
    display(IPImage(filename=STAGE3_REFINED_IMAGE_PATH))

# =====================================================================
# 3. INTERACTIVE 3D VIEWER (CONDITIONAL ON STAGE 4 SUCCESS)
# =====================================================================

def launch_trellis_viewer(target_glb_path, target_html_path):
    # ============================================================
    # CELL 3 — TRELLIS GLB INTERACTIVE VIEWER
    # ============================================================
    
    import sys
    import subprocess
    import os
    
    # ------------------------------------------------------------
    # Install viewer dependencies if missing
    # ------------------------------------------------------------
    
    def install_if_missing(package, import_name=None):
        import_name = import_name or package
    
        try:
            __import__(import_name)
            print(f"[OK] {package} already installed.")
        except ImportError:
            print(f"[INSTALL] Installing {package}...")
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package
            ])
            print(f"[OK] {package} installed.")
    
    
    install_if_missing("trimesh")
    install_if_missing("plotly")
    
    
    # ------------------------------------------------------------
    # Imports
    # ------------------------------------------------------------
    
    import numpy as np
    import trimesh
    import plotly.graph_objects as go
    
    
    # ------------------------------------------------------------
    # GLB path
    # ------------------------------------------------------------
    
    GLB_PATH = target_glb_path
    
    if not os.path.exists(GLB_PATH):
        raise FileNotFoundError(
            f"GLB not found:\n{GLB_PATH}\n\n"
            "Make sure Cell 2 finished successfully."
        )
    
    
    print()
    print("=" * 60)
    print("Loading TRELLIS 3D model...")
    print("=" * 60)
    
    
    # ------------------------------------------------------------
    # Load GLB
    # ------------------------------------------------------------
    
    scene_or_mesh = trimesh.load(
        GLB_PATH,
        force="scene"
    )
    
    
    # ------------------------------------------------------------
    # Extract mesh
    # ------------------------------------------------------------
    
    if isinstance(scene_or_mesh, trimesh.Scene):
    
        geometries = [
            g
            for g in scene_or_mesh.geometry.values()
            if isinstance(g, trimesh.Trimesh)
        ]
    
        if not geometries:
            raise RuntimeError(
                "No mesh geometry found inside GLB."
            )
    
        mesh = trimesh.util.concatenate(geometries)
    
    else:
    
        mesh = scene_or_mesh
    
    
    print("3D model loaded successfully.")
    print(f"Vertices : {len(mesh.vertices):,}")
    print(f"Faces    : {len(mesh.faces):,}")
    
    
    # ------------------------------------------------------------
    # Convert geometry to NumPy
    # ------------------------------------------------------------
    
    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)
    
    x = vertices[:, 0]
    y = vertices[:, 1]
    z = vertices[:, 2]
    
    i = faces[:, 0]
    j = faces[:, 1]
    k = faces[:, 2]
    
    
    # ------------------------------------------------------------
    # Try to extract vertex colors
    # ------------------------------------------------------------
    
    vertex_colors = None
    
    try:
    
        colors = mesh.visual.vertex_colors
    
        if (
            colors is not None
            and len(colors) == len(vertices)
        ):
            vertex_colors = colors[:, :3]
    
    except Exception:
        pass
    
    
    # ------------------------------------------------------------
    # Create Plotly 3D viewer
    # ------------------------------------------------------------
    
    if vertex_colors is not None:
    
        print("Vertex colors detected.")
    
        fig = go.Figure(
            data=[
                go.Mesh3d(
                    x=x,
                    y=y,
                    z=z,
                    i=i,
                    j=j,
                    k=k,
                    vertexcolor=vertex_colors,
                    flatshading=False,
                    opacity=1.0
                )
            ]
        )
    
    else:
    
        print("No vertex colors detected.")
        print("Displaying mesh in light gray.")
    
        fig = go.Figure(
            data=[
                go.Mesh3d(
                    x=x,
                    y=y,
                    z=z,
                    i=i,
                    j=j,
                    k=k,
                    color="lightgray",
                    flatshading=False,
                    opacity=1.0
                )
            ]
        )
    
    
    # ------------------------------------------------------------
    # Viewer layout
    # ------------------------------------------------------------
    
    fig.update_layout(
    
        title="TRELLIS Interactive 3D Mesh",
    
        scene=dict(
    
            xaxis=dict(
                visible=False
            ),
    
            yaxis=dict(
                visible=False
            ),
    
            zaxis=dict(
                visible=False
            ),
    
            aspectmode="data",
    
            bgcolor="white",
    
            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.1
                )
            )
        ),
    
        paper_bgcolor="white",
    
        margin=dict(
            l=0,
            r=0,
            t=40,
            b=0
        ),
    
        height=800
    )
    
    
    # ------------------------------------------------------------
    # Save HTML viewer
    # ------------------------------------------------------------
    
    HTML_PATH = target_html_path
    
    fig.write_html(
        HTML_PATH,
        full_html=True,
        include_plotlyjs=True
    )
    
    
    # ------------------------------------------------------------
    # Display viewer directly
    # ------------------------------------------------------------
    
    print()
    print("=" * 60)
    print("Viewer saved successfully:")
    print(HTML_PATH)
    print("=" * 60)
    print()
    
    fig.show()

# Execute viewer only if Stage 4 GLB is confirmed for the current execution
if crit7 == "PASS":
    print("\n" + "=" * 60)
    print("Launching Interactive 3D Mesh Viewer for Current Execution GLB...")
    print("=" * 60)
    launch_trellis_viewer(GLB_PATH, HTML_PATH)
    print("\n✓ [Cell 5 Output] 3D mesh viewer rendered and saved to:", HTML_PATH)
else:
    print(f"\n[3D Viewer Notice] 3D mesh viewer skipped: '{os.path.basename(GLB_PATH)}' is [{crit7}].")
    print("Cell 4 must complete successfully during the current execution to launch the interactive 3D viewer.")